In [ ]:
# 导入必要的库
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import traceback
import seaborn as sns
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import h5py
import json
from datetime import datetime
from tqdm import tqdm
import pickle
import warnings
import logging
from scipy import linalg

# 设置中英双语日志系统
class BilingualLogger:
    def __init__(self, log_dir="logs", log_name=None):
        # 创建日志目录
        os.makedirs(log_dir, exist_ok=True)
        
        # 设置日志名称
        if log_name is None:
            log_name = f"pseudoinverse_experiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
        
        self.logger = logging.getLogger("pseudoinverse_experiment")
        self.logger.setLevel(logging.INFO)
        
        # 文件处理器
        file_handler = logging.FileHandler(os.path.join(log_dir, log_name), encoding='utf-8')
        file_handler.setLevel(logging.INFO)
        
        # 控制台处理器
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)
        
        # 设置格式
        formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
        file_handler.setFormatter(formatter)
        console_handler.setFormatter(formatter)
        
        # 添加处理器
        self.logger.addHandler(file_handler)
        self.logger.addHandler(console_handler)
    
    def info(self, msg_cn, msg_en=None):
        """记录信息级别的日志"""
        if msg_en is None:
            self.logger.info(f"{msg_cn}")
        else:
            self.logger.info(f"{msg_cn} | {msg_en}")
    
    def warning(self, msg_cn, msg_en=None):
        """记录警告级别的日志"""
        if msg_en is None:
            self.logger.warning(f"{msg_cn}")
        else:
            self.logger.warning(f"{msg_cn} | {msg_en}")
    
    def error(self, msg_cn, msg_en=None):
        """记录错误级别的日志"""
        if msg_en is None:
            self.logger.error(f"{msg_cn}")
        else:
            self.logger.error(f"{msg_cn} | {msg_en}")

# 设置显示风格
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style="whitegrid")

# 设置中文字体支持（如果需要在图表中显示中文）
# plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
# plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 忽略特定警告
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("库导入完成，环境设置就绪")

In [ ]:
# 特征选择模块
class FeatureSelector:
    """使用LASSO或弹性网络进行特征选择的类"""
    
    def __init__(self, method='lasso', selection_mode='threshold', selection_threshold=0.01,
                 max_features=100, l1_ratio=1.0, cv_folds=5, random_state=42,
                 scaling_before_selection=True, selection_metric='coefficient',
                 logger=None):
        """
        初始化特征选择器
        
        参数:
            method: 特征选择方法，'lasso'或'elastic_net'
            selection_mode: 特征选择模式，'threshold'（基于阈值）或'fixed'（固定数量）
            selection_threshold: 特征选择阈值，用于'threshold'模式
            max_features: 最大特征数量，用于'fixed'模式
            l1_ratio: 弹性网络的L1比例（1.0为纯LASSO）
            cv_folds: 交叉验证折数，用于评估特征选择稳定性
            random_state: 随机种子
            scaling_before_selection: 是否在特征选择前进行标准化
            selection_metric: 特征重要性度量，'coefficient'或'importance'
            logger: 日志记录器
        """
        self.method = method
        self.selection_mode = selection_mode
        self.selection_threshold = selection_threshold
        self.max_features = max_features
        self.l1_ratio = l1_ratio
        self.cv_folds = cv_folds
        self.random_state = random_state
        self.scaling_before_selection = scaling_before_selection
        self.selection_metric = selection_metric
        self.logger = logger if logger else BilingualLogger()
        self.selection_frequency = None
        self.jaccard_matrix = None
        
        # 特征选择模型
        self.model = None
        # 所选特征的索引
        self.selected_indices = None
        # 特征重要性
        self.feature_importance = None
        # 标准化器
        self.scaler = None
        
        self.logger.info(f"特征选择器初始化: 方法={method}, 选择模式={selection_mode}, "
                      f"阈值/最大特征数={selection_threshold if selection_mode=='threshold' else max_features}",
                      f"Feature selector initialized: method={method}, selection_mode={selection_mode}, "
                      f"threshold/max_features={selection_threshold if selection_mode=='threshold' else max_features}")
    
    def fit(self, X, y, class_weight=None):
        """
        拟合特征选择模型并选择特征
        
        参数:
            X: 特征矩阵，形状为(n_samples, n_features)
            y: 标签，形状为(n_samples,)
            class_weight: 类别权重，用于多分类问题
            
        返回:
            self: 特征选择器自身
        """
        from sklearn.linear_model import Lasso, ElasticNet, LogisticRegression
        from sklearn.preprocessing import StandardScaler
        from sklearn.multiclass import OneVsRestClassifier
        import time
        
        start_time = time.time()
        self.logger.info(f"开始特征选择, 输入特征维度: {X.shape[1]}", 
                      f"Starting feature selection, input dimension: {X.shape[1]}")
        
        # 标准化特征
        if self.scaling_before_selection:
            self.logger.info("应用标准化预处理", "Applying standardization preprocessing")
            self.scaler = StandardScaler()
            X = self.scaler.fit_transform(X)
        
        # 创建模型
        if self.method == 'lasso':
            # 对于多分类问题，使用OneVsRestClassifier包装LogisticRegression
            if len(np.unique(y)) > 2:
                base_model = LogisticRegression(
                    penalty='l1', solver='liblinear', C=1.0/self.selection_threshold,
                    random_state=self.random_state, class_weight=class_weight)
                
                self.model = OneVsRestClassifier(base_model)
            else:
                # 二分类问题
                self.model = LogisticRegression(
                    penalty='l1', solver='liblinear', C=1.0/self.selection_threshold,
                    random_state=self.random_state, class_weight=class_weight)
        
        elif self.method == 'elastic_net':
            # 对于多分类问题
            if len(np.unique(y)) > 2:
                base_model = LogisticRegression(
                    penalty='elasticnet', solver='saga', C=1.0/self.selection_threshold,
                    l1_ratio=self.l1_ratio, random_state=self.random_state, 
                    class_weight=class_weight)
                
                self.model = OneVsRestClassifier(base_model)
            else:
                # 二分类问题
                self.model = LogisticRegression(
                    penalty='elasticnet', solver='saga', C=1.0/self.selection_threshold,
                    l1_ratio=self.l1_ratio, random_state=self.random_state,
                    class_weight=class_weight)
        
        # 拟合模型
        self.model.fit(X, y)
        
        # 获取特征重要性
        if hasattr(self.model, 'coef_'):
            coefs = self.model.coef_
        else:
            # 对于OneVsRestClassifier，特征系数在estimators_中
            coefs = np.vstack([est.coef_ for est in self.model.estimators_])
        
        # 计算特征重要性
        if self.selection_metric == 'coefficient':
            # 使用系数绝对值的平均值作为特征重要性
            self.feature_importance = np.mean(np.abs(coefs), axis=0)
        else:
            # TODO: 实现其他特征重要性度量，如SHAP值
            self.feature_importance = np.mean(np.abs(coefs), axis=0)
        
        # 选择特征
        if self.selection_mode == 'threshold':
            # 根据阈值选择特征
            self.selected_indices = np.where(self.feature_importance > self.selection_threshold)[0]
        else:
            # 固定数量模式
            if self.max_features < X.shape[1]:
                # 选择top-k特征
                self.selected_indices = np.argsort(self.feature_importance)[-self.max_features:]
            else:
                # 如果max_features大于等于特征数，保留所有特征
                self.selected_indices = np.arange(X.shape[1])
        
        # 确保索引已排序
        self.selected_indices = np.sort(self.selected_indices)
        
        elapsed_time = time.time() - start_time
        self.logger.info(f"特征选择完成，选择了 {len(self.selected_indices)}/{X.shape[1]} 个特征，"
                      f"耗时 {elapsed_time:.2f} 秒",
                      f"Feature selection completed, selected {len(self.selected_indices)}/{X.shape[1]} "
                      f"features in {elapsed_time:.2f} seconds")
        
        return self
    
    def transform(self, X):
        """
        使用选定的特征转换数据
        
        参数:
            X: 特征矩阵，形状为(n_samples, n_features)
            
        返回:
            X_selected: 选择特征后的数据，形状为(n_samples, n_selected_features)
        """
        if self.selected_indices is None:
            raise ValueError("模型尚未拟合，请先调用fit方法")
        
        # 应用相同的标准化
        if self.scaling_before_selection and self.scaler is not None:
            X = self.scaler.transform(X)
        
        # 选择特征
        return X[:, self.selected_indices]
    
    def fit_transform(self, X, y, class_weight=None):
        """
        拟合特征选择模型并转换数据
        
        参数:
            X: 特征矩阵，形状为(n_samples, n_features)
            y: 标签，形状为(n_samples,)
            class_weight: 类别权重，用于多分类问题
            
        返回:
            X_selected: 选择特征后的数据，形状为(n_samples, n_selected_features)
        """
        self.fit(X, y, class_weight)
        return self.transform(X)
    
    def get_support(self):
        """
        获取所选特征的布尔掩码
        
        返回:
            support: 布尔掩码，指示每个特征是否被选择
        """
        if self.selected_indices is None:
            raise ValueError("模型尚未拟合，请先调用fit方法")
        
        support = np.zeros(self.feature_importance.shape[0], dtype=bool)
        support[self.selected_indices] = True
        return support
    
    def get_feature_importance(self):
        """
        获取特征重要性
        
        返回:
            importance: 特征重要性向量
        """
        if self.feature_importance is None:
            raise ValueError("模型尚未拟合，请先调用fit方法")
        
        return self.feature_importance
    
    def get_selected_indices(self):
        """
        获取所选特征的索引
        
        返回:
            indices: 所选特征的索引
        """
        if self.selected_indices is None:
            raise ValueError("模型尚未拟合，请先调用fit方法")
        
        return self.selected_indices
    
    def evaluate_stability(self, X, y, n_splits=None):
        """
        评估特征选择的稳定性
        
        参数:
            X: 特征矩阵，形状为(n_samples, n_features)
            y: 标签，形状为(n_samples,)
            n_splits: 交叉验证折数，如果为None则使用初始化时设置的值
            
        返回:
            stability_metrics: 包含稳定性指标的字典
        """
        from sklearn.model_selection import KFold
        from sklearn.metrics import jaccard_score
        
        if n_splits is None:
            n_splits = self.cv_folds
        
        self.logger.info(f"评估特征选择稳定性，使用{n_splits}折交叉验证", 
                      f"Evaluating feature selection stability using {n_splits}-fold CV")
        
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=self.random_state)
        selected_features_masks = []
        
        # 在每个折上进行特征选择
        for train_idx, _ in kf.split(X):
            X_train, y_train = X[train_idx], y[train_idx]
            
            # 创建相同参数的选择器
            selector = FeatureSelector(
                method=self.method,
                selection_mode=self.selection_mode,
                selection_threshold=self.selection_threshold,
                max_features=self.max_features,
                l1_ratio=self.l1_ratio
            )
            
            # 拟合选择器
            selector.fit(X_train, y_train)
            
            # 获取所选特征的掩码
            mask = selector.get_support()
            selected_features_masks.append(mask)
        
        # 计算Jaccard相似度矩阵
        n_folds = len(selected_features_masks)
        jaccard_matrix = np.zeros((n_folds, n_folds))
        
        for i in range(n_folds):
            for j in range(i, n_folds):
                if i == j:
                    jaccard_matrix[i, j] = 1.0
                else:
                    similarity = jaccard_score(
                        selected_features_masks[i], 
                        selected_features_masks[j], 
                        average='binary'
                    )
                    jaccard_matrix[i, j] = similarity
                    jaccard_matrix[j, i] = similarity
        
        # 计算平均Jaccard相似度
        avg_jaccard = np.mean(jaccard_matrix[np.triu_indices(n_folds, k=1)])
        
        # 计算特征选择频率
        selection_frequency = np.mean(selected_features_masks, axis=0)
        
        # 计算稳定性指标
        stability_metrics = {
            'avg_jaccard_similarity': avg_jaccard,
            'selection_frequency': selection_frequency,
            'jaccard_matrix': jaccard_matrix
        }
        
        self.logger.info(f"特征选择稳定性评估完成，平均Jaccard相似度: {avg_jaccard:.4f}",
                      f"Feature selection stability evaluation completed, "
                      f"average Jaccard similarity: {avg_jaccard:.4f}")
        
        return stability_metrics
    
    def save(self, filepath):
        """
        保存特征选择器到文件
        
        参数:
            filepath: 保存路径
        """
        model_data = {
            'method': self.method,
            'selection_mode': self.selection_mode,
            'selection_threshold': self.selection_threshold,
            'max_features': self.max_features,
            'l1_ratio': self.l1_ratio,
            'cv_folds': self.cv_folds,
            'random_state': self.random_state,
            'scaling_before_selection': self.scaling_before_selection,
            'selection_metric': self.selection_metric,
            'selected_indices': self.selected_indices,
            'feature_importance': self.feature_importance
        }
        
        with open(filepath, 'wb') as f:
            pickle.dump(model_data, f)
        
        self.logger.info(f"特征选择器已保存到: {filepath}", f"Feature selector saved to: {filepath}")
    
    def load(self, filepath):
        """
        从文件加载特征选择器
        
        参数:
            filepath: 加载路径
        """
        with open(filepath, 'rb') as f:
            model_data = pickle.load(f)
        
        self.method = model_data['method']
        self.selection_mode = model_data['selection_mode']
        self.selection_threshold = model_data['selection_threshold']
        self.max_features = model_data['max_features']
        self.l1_ratio = model_data['l1_ratio']
        self.cv_folds = model_data['cv_folds']
        self.random_state = model_data['random_state']
        self.scaling_before_selection = model_data['scaling_before_selection']
        self.selection_metric = model_data['selection_metric']
        self.selected_indices = model_data['selected_indices']
        self.feature_importance = model_data['feature_importance']
        
        self.logger.info(f"特征选择器已从 {filepath} 加载", f"Feature selector loaded from: {filepath}")
        
        return self

In [ ]:
# 特征选择可视化函数
def visualize_feature_selection(selector, feature_names=None, save_path=None):
    """
    可视化特征选择结果
    
    参数:
        selector: 已拟合的特征选择器
        feature_names: 特征名称列表
        save_path: 图表保存路径
    """
    if selector.feature_importance is None:
        print("特征选择器尚未拟合或没有特征重要性信息")
        return
    
    # 获取特征重要性和选择掩码
    importance = selector.get_feature_importance()
    selected = selector.get_support()
    
    # 如果没有提供特征名称，使用索引
    if feature_names is None:
        feature_names = [f"Feature {i}" for i in range(len(importance))]
    
    # 确保特征名称长度匹配
    if len(feature_names) != len(importance):
        feature_names = [f"Feature {i}" for i in range(len(importance))]
    
    # 创建一个包含3个子图的图表
    plt.figure(figsize=(18, 12))
    
    # 1. 特征重要性分布图
    plt.subplot(2, 2, 1)
    plt.hist(importance, bins=50, alpha=0.7)
    plt.axvline(x=selector.selection_threshold if selector.selection_mode == 'threshold' else 
               importance[np.argsort(importance)[-selector.max_features]], 
               color='r', linestyle='--', 
               label=f"Selection Threshold: {selector.selection_threshold}" if selector.selection_mode == 'threshold' else 
               f"Top {selector.max_features} Features")
    plt.xlabel('Feature Importance')
    plt.ylabel('Frequency')
    plt.title('Distribution of Feature Importance')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # 2. 所选特征在原始特征空间中的位置
    plt.subplot(2, 2, 2)
    plt.scatter(range(len(importance)), importance, alpha=0.5, 
               c=['blue' if s else 'gray' for s in selected])
    plt.xlabel('Feature Index')
    plt.ylabel('Feature Importance')
    plt.title('Selected Features in Feature Space')
    # 如果特征太多，只显示部分刻度
    if len(importance) > 20:
        tick_step = len(importance) // 10
        plt.xticks(range(0, len(importance), tick_step))
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # 3. 前N个重要特征
    top_n = min(20, np.sum(selected))
    top_indices = np.argsort(importance)[-top_n:][::-1]
    top_importance = importance[top_indices]
    top_names = [feature_names[i] for i in top_indices]
    
    plt.subplot(2, 2, 3)
    plt.barh(range(len(top_names)), top_importance, align='center')
    plt.yticks(range(len(top_names)), top_names)
    plt.xlabel('Feature Importance')
    plt.title(f'Top {top_n} Most Important Features')
    plt.grid(True, axis='x', linestyle='--', alpha=0.7)
    
    # 4. 选择频率分布（如果有）
    if hasattr(selector, 'selection_frequency'):
        plt.subplot(2, 2, 4)
        plt.hist(selector.selection_frequency, bins=10, alpha=0.7)
        plt.xlabel('Selection Frequency')
        plt.ylabel('Number of Features')
        plt.title('Feature Selection Stability')
        plt.grid(True, linestyle='--', alpha=0.7)
    else:
        # 或者替代图：选择前后的特征数量
        plt.subplot(2, 2, 4)
        plt.bar(['Original', 'Selected'], [len(importance), np.sum(selected)])
        plt.ylabel('Number of Features')
        plt.title('Feature Reduction')
        for i, v in enumerate([len(importance), np.sum(selected)]):
            plt.text(i, v + 0.1, str(v), ha='center')
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

def visualize_feature_stability(stability_metrics, save_path=None):
    """
    可视化特征选择稳定性
    
    参数:
        stability_metrics: 包含稳定性指标的字典
        save_path: 图表保存路径
    """
    jaccard_matrix = stability_metrics['jaccard_matrix']
    selection_frequency = stability_metrics['selection_frequency']
    
    plt.figure(figsize=(15, 6))
    
    # 1. Jaccard相似度热图
    plt.subplot(1, 2, 1)
    sns.heatmap(jaccard_matrix, annot=True, cmap='Blues', vmin=0, vmax=1)
    plt.title('Jaccard Similarity Between Folds')
    plt.xlabel('Fold')
    plt.ylabel('Fold')
    
    # 2. 特征选择频率分布
    plt.subplot(1, 2, 2)
    plt.hist(selection_frequency, bins=10, alpha=0.7)
    plt.xlabel('Selection Frequency Across Folds')
    plt.ylabel('Number of Features')
    plt.title('Feature Selection Stability')
    plt.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

def visualize_selected_feature_performance(results_before, results_after, save_path=None):
    """
    可视化特征选择前后的性能对比
    
    参数:
        results_before: 特征选择前的评估结果
        results_after: 特征选择后的评估结果
        save_path: 图表保存路径
    """
    # 提取数据
    datasets = list(results_before.keys())
    metrics = ['accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted', 'kappa']
    metric_names = ['Accuracy', 'Balanced Accuracy', 'Macro F1', 'Weighted F1', 'Kappa']
    
    plt.figure(figsize=(15, 10))
    
    # 为每个数据集绘制性能对比图
    for i, dataset in enumerate(datasets):
        if dataset in ['train_time', 'evaluation_time']:
            continue
            
        plt.subplot(len(datasets), 1, i+1)
        
        # 准备数据
        before_values = [results_before[dataset][m] for m in metrics]
        after_values = [results_after[dataset][m] for m in metrics]
        
        # 绘制柱状图
        x = np.arange(len(metrics))
        width = 0.35
        
        plt.bar(x - width/2, before_values, width, label='Before Feature Selection')
        plt.bar(x + width/2, after_values, width, label='After Feature Selection')
        
        plt.xlabel('Metrics')
        plt.ylabel('Score')
        plt.title(f'Performance Comparison on {dataset.capitalize()} Set')
        plt.xticks(x, metric_names)
        plt.ylim(0, 1.0)
        plt.legend()
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        
        # 添加数值标签
        for j, (before, after) in enumerate(zip(before_values, after_values)):
            plt.text(j - width/2, before + 0.01, f"{before:.3f}", ha='center', va='bottom', fontsize=8)
            plt.text(j + width/2, after + 0.01, f"{after:.3f}", ha='center', va='bottom', fontsize=8)
            
            # 计算变化百分比
            if before > 0:
                change_pct = (after - before) / before * 100
                color = 'green' if change_pct >= 0 else 'red'
                plt.text(j, max(before, after) + 0.05, f"{change_pct:+.1f}%", 
                        ha='center', va='bottom', fontsize=9, color=color)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

In [ ]:
# 定义数据加载类
class BrainVoxelDataLoader:
    """脑体素数据加载类，处理训练、测试和验证数据"""
    
    def __init__(self, train_dir, test_dir, val_dir, logger=None):
        """
        初始化数据加载器
        
        参数:
            train_dir: 训练数据目录
            test_dir: 测试数据目录
            val_dir: 验证数据目录
            logger: 日志记录器
        """
        self.train_dir = train_dir
        self.test_dir = test_dir
        self.val_dir = val_dir
        self.logger = logger if logger else BilingualLogger()
        
        # 存储加载的数据
        self.train_samples = None
        self.train_labels = None
        self.test_samples = None
        self.test_labels = None
        self.val_samples = None
        self.val_labels = None
        
        # 预处理模型
        self.pca_model = None
        self.scaler = None
        
        self.logger.info("数据加载器初始化完成", "Data loader initialized")
    
    def _load_data_from_dir(self, directory, desc="加载数据"):
        """从指定目录加载数据"""
        # 创建采样器
        sampler = BrainVoxelSampler(directory)
        valid_labels = sampler.valid_labels
        
        # 加载每个标签的数据
        all_samples = []
        all_labels = []
        
        for label_id in tqdm(valid_labels, desc=desc):
            file_path = sampler.get_file_path(label_id)
            if file_path:
                samples = np.load(file_path)
                labels = np.ones(len(samples)) * label_id
                all_samples.append(samples)
                all_labels.append(labels)
        
        # 合并数据
        if all_samples:
            all_samples = np.vstack(all_samples)
            all_labels = np.concatenate(all_labels)
            return all_samples, all_labels
        else:
            return np.array([]), np.array([])
    
    def load_all_data(self):
        """加载所有数据集"""
        self.logger.info("开始加载所有数据集", "Start loading all datasets")
        
        # 加载训练集
        self.train_samples, self.train_labels = self._load_data_from_dir(
            self.train_dir, "加载训练集数据")
        
        # 加载测试集
        self.test_samples, self.test_labels = self._load_data_from_dir(
            self.test_dir, "加载测试集数据")
        
        # 加载验证集
        self.val_samples, self.val_labels = self._load_data_from_dir(
            self.val_dir, "加载验证集数据")
        
        # 打印数据集统计信息
        self._print_dataset_stats()
        
        return {
            'train_samples': self.train_samples,
            'train_labels': self.train_labels,
            'test_samples': self.test_samples,
            'test_labels': self.test_labels,
            'val_samples': self.val_samples,
            'val_labels': self.val_labels
        }
    
    def _print_dataset_stats(self):
        """打印数据集统计信息"""
        self.logger.info(
            f"数据集统计信息:\n"
            f"训练集: {len(self.train_labels)} 个样本\n"
            f"测试集: {len(self.test_labels)} 个样本\n"
            f"验证集: {len(self.val_labels)} 个样本",
            
            f"Dataset statistics:\n"
            f"Training set: {len(self.train_labels)} samples\n"
            f"Test set: {len(self.test_labels)} samples\n"
            f"Validation set: {len(self.val_labels)} samples"
        )
    
    def preprocess_data(self, apply_pca=False, n_components=50, 
                       normalization='standard', class_balance=False,
                       target_samples=1000, random_state=42):
        """
        预处理数据
        
        参数:
            apply_pca: 是否应用PCA降维
            n_components: PCA组件数量
            normalization: 标准化方法，'standard'或'minmax'或None
            class_balance: 是否进行类别平衡
            target_samples: 每个类别的目标样本数
            random_state: 随机种子
        """
        processed_data = {}
        
        # 复制原始数据
        train_X = self.train_samples.copy()
        test_X = self.test_samples.copy()
        val_X = self.val_samples.copy()
        
        train_y = self.train_labels.copy()
        test_y = self.test_labels.copy()
        val_y = self.val_labels.copy()
        
        # 1. 应用PCA降维
        if apply_pca and n_components > 0:
            self.logger.info(f"应用PCA降维至{n_components}个组件", 
                            f"Applying PCA reduction to {n_components} components")
            
            # 创建并拟合PCA模型
            self.pca_model = PCA(n_components=n_components, random_state=random_state)
            train_X = self.pca_model.fit_transform(train_X)
            
            # 转换测试集和验证集
            test_X = self.pca_model.transform(test_X)
            val_X = self.pca_model.transform(val_X)
            
            # 记录解释方差
            explained_variance = np.sum(self.pca_model.explained_variance_ratio_)
            self.logger.info(f"PCA解释方差: {explained_variance:.4f}",
                            f"PCA explained variance: {explained_variance:.4f}")
        
        # 2. 应用标准化
        if normalization == 'standard':
            self.logger.info("应用标准化(Z-score)", "Applying standardization (Z-score)")
            self.scaler = StandardScaler()
            train_X = self.scaler.fit_transform(train_X)
            test_X = self.scaler.transform(test_X)
            val_X = self.scaler.transform(val_X)
            
        elif normalization == 'minmax':
            self.logger.info("应用归一化(MinMax)", "Applying normalization (MinMax)")
            self.scaler = MinMaxScaler()
            train_X = self.scaler.fit_transform(train_X)
            test_X = self.scaler.transform(test_X)
            val_X = self.scaler.transform(val_X)
        
        # 3. 类别平衡（仅针对训练集）
        if class_balance:
            self.logger.info(f"应用类别平衡，目标每类{target_samples}个样本",
                           f"Applying class balancing, target {target_samples} samples per class")
            
            balanced_X = []
            balanced_y = []
            
            # 获取唯一类别
            unique_classes = np.unique(train_y)
            
            # 对每个类别进行平衡
            for cls in unique_classes:
                cls_idx = np.where(train_y == cls)[0]
                cls_samples = train_X[cls_idx]
                cls_labels = train_y[cls_idx]
                
                # 如果样本数量超过目标，随机下采样
                if len(cls_idx) > target_samples:
                    np.random.seed(random_state)
                    select_idx = np.random.choice(
                        len(cls_idx), target_samples, replace=False)
                    cls_samples = cls_samples[select_idx]
                    cls_labels = cls_labels[select_idx]
                
                # 如果样本数量不足目标，过采样
                elif len(cls_idx) < target_samples and len(cls_idx) > 0:
                    # 简单的复制采样
                    n_copies = target_samples // len(cls_idx)
                    remainder = target_samples % len(cls_idx)
                    
                    # 复制整数倍
                    if n_copies > 0:
                        cls_samples = np.repeat(cls_samples, n_copies, axis=0)
                        cls_labels = np.repeat(cls_labels, n_copies)
                    
                    # 处理余数
                    if remainder > 0:
                        np.random.seed(random_state)
                        extra_idx = np.random.choice(
                            len(cls_idx), remainder, replace=False)
                        cls_samples = np.vstack([cls_samples, train_X[cls_idx[extra_idx]]])
                        cls_labels = np.concatenate([cls_labels, train_y[cls_idx[extra_idx]]])
                
                balanced_X.append(cls_samples)
                balanced_y.append(cls_labels)
            
            # 合并所有平衡后的类别
            train_X = np.vstack(balanced_X)
            train_y = np.concatenate(balanced_y)
            
            # 打乱数据
            np.random.seed(random_state)
            shuffle_idx = np.random.permutation(len(train_y))
            train_X = train_X[shuffle_idx]
            train_y = train_y[shuffle_idx]
            
            self.logger.info(f"类别平衡后训练集大小: {len(train_y)}", 
                           f"Training set size after balancing: {len(train_y)}")
        
        # 保存处理后的数据
        processed_data['train_X'] = train_X
        processed_data['train_y'] = train_y
        processed_data['test_X'] = test_X
        processed_data['test_y'] = test_y
        processed_data['val_X'] = val_X
        processed_data['val_y'] = val_y
        
        return processed_data

# 实现脑体素采样器（从您现有代码中复用）
class BrainVoxelSampler:
    """脑体素数据采样器，提供多种采样策略"""
    
    def __init__(self, data_dir):
        """
        初始化采样器
        
        参数:
            data_dir: 数据集目录
        """
        self.data_dir = data_dir
        self.label_info = self._load_label_index()
        self.valid_labels = [label for label, info in self.label_info.items() if info['count'] > 0]
    
    def _load_label_index(self):
        """加载标签索引文件"""
        index_file = os.path.join(self.data_dir, "label_index.txt")
        label_info = {}
        
        with open(index_file, 'r') as f:
            # 跳过表头
            next(f)
            for line in f:
                parts = line.strip().split(',')
                if len(parts) >= 3:
                    label_id = int(parts[0])
                    voxel_count = int(parts[1])
                    filename = parts[2] if parts[2] else None
                    label_info[label_id] = {'count': voxel_count, 'filename': filename}
        
        return label_info
    
    def get_file_path(self, label_id):
        """获取指定标签的文件路径"""
        if label_id not in self.label_info:
            return None
        
        filename = self.label_info[label_id]['filename']
        if not filename:
            return None
            
        return os.path.join(self.data_dir, filename)

In [ ]:
# 扩展数据预处理函数以支持特征选择
def preprocess_data_with_feature_selection(self, apply_pca=False, n_components=50, 
                                          normalization='standard', class_balance=False,
                                          target_samples=1000, random_state=42,
                                          feature_selection=None, selection_mode='threshold',
                                          selection_threshold=0.01, max_features=100,
                                          l1_ratio=1.0, cv_folds=5,
                                          scaling_before_selection=True,
                                          selection_metric='coefficient'):
    """
    预处理数据，包括可选的特征选择
    
    参数:
        apply_pca: 是否应用PCA降维
        n_components: PCA组件数量
        normalization: 标准化方法，'standard'或'minmax'或None
        class_balance: 是否进行类别平衡
        target_samples: 每个类别的目标样本数
        random_state: 随机种子
        feature_selection: 特征选择方法，None, 'lasso'或'elastic_net'
        selection_mode: 特征选择模式，'threshold'或'fixed'
        selection_threshold: 特征选择阈值
        max_features: 最大特征数量
        l1_ratio: 弹性网络的L1比例
        cv_folds: 交叉验证折数
        scaling_before_selection: 是否在特征选择前进行标准化
        selection_metric: 特征重要性度量
        
    返回:
        预处理后的数据字典
    """
    # 首先进行常规预处理
    processed_data = self.preprocess_data(
        apply_pca=apply_pca,
        n_components=n_components,
        normalization=normalization,
        class_balance=class_balance,
        target_samples=target_samples,
        random_state=random_state
    )
    
    # 如果不需要特征选择，直接返回
    if feature_selection is None:
        return processed_data
    
    self.logger.info(f"应用特征选择: 方法={feature_selection}, 模式={selection_mode}",
                  f"Applying feature selection: method={feature_selection}, mode={selection_mode}")
    
    # 创建特征选择器
    selector = FeatureSelector(
        method=feature_selection,
        selection_mode=selection_mode,
        selection_threshold=selection_threshold,
        max_features=max_features,
        l1_ratio=l1_ratio,
        cv_folds=cv_folds,
        random_state=random_state,
        scaling_before_selection=scaling_before_selection,
        selection_metric=selection_metric,
        logger=self.logger
    )
    
    # 拟合特征选择器并转换训练数据
    train_X_selected = selector.fit_transform(
        processed_data['train_X'], 
        processed_data['train_y']
    )
    
    # 转换测试和验证数据
    test_X_selected = selector.transform(processed_data['test_X'])
    val_X_selected = selector.transform(processed_data['val_X'])
    
    # 更新处理后的数据
    processed_data['train_X'] = train_X_selected
    processed_data['test_X'] = test_X_selected
    processed_data['val_X'] = val_X_selected
    processed_data['feature_selector'] = selector
    
    # 获取所选特征的索引和数量
    selected_indices = selector.get_selected_indices()
    self.logger.info(f"特征选择完成，选择了 {len(selected_indices)} 个特征",
                  f"Feature selection completed, selected {len(selected_indices)} features")
    
    return processed_data

# 为BrainVoxelDataLoader类添加该方法
BrainVoxelDataLoader.preprocess_data_with_feature_selection = preprocess_data_with_feature_selection

In [ ]:
class PseudoInverseModel:
    """使用伪逆方法的线性模型类"""
    
    def __init__(self, num_classes=102, logger=None):
        """
        初始化伪逆模型
        
        参数:
            num_classes: 类别数量
            logger: 日志记录器
        """
        self.num_classes = num_classes
        self.weights = None  # 权重矩阵
        self.bias = None     # 偏置向量
        self.feature_importance = None  # 特征重要性
        self.logger = logger if logger else BilingualLogger()
        
        self.logger.info(f"初始化伪逆模型，类别数：{num_classes}", 
                       f"Initialized pseudo-inverse model with {num_classes} classes")
    
    def fit(self, X, y, regularization=None, alpha=0.0):
        """
        使用伪逆方法拟合线性模型
        
        参数:
            X: 特征矩阵，形状为(n_samples, n_features)
            y: 标签，形状为(n_samples,)
            regularization: 正则化方法，None, 'l2'或'truncated'
            alpha: 正则化强度，用于L2正则化
        """
        self.logger.info("开始拟合伪逆模型", "Starting to fit pseudo-inverse model")
        start_time = time.time()
        
        # 将标签转换为独热编码
        y_one_hot = self._to_one_hot(y)
        
        # 添加偏置项
        X_bias = np.hstack((X, np.ones((X.shape[0], 1))))
        
        # 根据正则化方法计算伪逆
        if regularization == 'l2':
            # Tikhonov正则化（L2）
            self.logger.info(f"应用L2正则化，alpha={alpha}", 
                           f"Applying L2 regularization with alpha={alpha}")
            n_features = X_bias.shape[1]
            identity = np.eye(n_features)
            identity[-1, -1] = 0  # 不对偏置项正则化
            
            # 解析解：W = (X^T X + alpha*I)^(-1) X^T y
            XTX = X_bias.T @ X_bias
            XTX_reg = XTX + alpha * identity
            weights = np.linalg.solve(XTX_reg, X_bias.T @ y_one_hot)
        
        elif regularization == 'truncated':
            # 截断SVD伪逆
            self.logger.info(f"应用截断SVD伪逆，alpha={alpha}", 
                           f"Applying truncated SVD with threshold={alpha}")
            
            # 使用SVD计算伪逆
            U, s, Vh = np.linalg.svd(X_bias, full_matrices=False)
            
            # 截断小于阈值的奇异值
            s_threshold = alpha * max(s)
            s_inv = np.array([1/si if si > s_threshold else 0 for si in s])
            
            # 计算伪逆
            pinv_X = (Vh.T * s_inv) @ U.T
            weights = pinv_X @ y_one_hot
        
        else:
            # 标准伪逆
            self.logger.info("应用标准伪逆", "Applying standard pseudo-inverse")
            pinv_X = np.linalg.pinv(X_bias)
            weights = pinv_X @ y_one_hot
        
        # 分离权重和偏置
        self.weights = weights[:-1, :]
        self.bias = weights[-1, :]
        
        # 计算特征重要性（每个特征权重的平均绝对值）
        self.feature_importance = np.mean(np.abs(self.weights), axis=1)
        
        # 记录拟合时间
        elapsed_time = time.time() - start_time
        self.logger.info(f"伪逆模型拟合完成，耗时 {elapsed_time:.2f} 秒", 
                       f"Pseudo-inverse model fitting completed in {elapsed_time:.2f} seconds")
        
        # 计算训练误差
        y_pred = self.predict(X)
        accuracy = accuracy_score(y, y_pred)
        self.logger.info(f"训练集准确率: {accuracy:.4f}", f"Training accuracy: {accuracy:.4f}")
        
        return self
    
    def predict(self, X):
        """
        使用拟合的模型进行预测
        
        参数:
            X: 特征矩阵，形状为(n_samples, n_features)
            
        返回:
            预测的类别，形状为(n_samples,)
        """
        if self.weights is None or self.bias is None:
            raise ValueError("模型尚未拟合，请先调用fit方法")
        
        # 计算线性输出
        logits = X @ self.weights + self.bias
        
        # 返回最大概率的类别
        return np.argmax(logits, axis=1)
    
    def predict_proba(self, X):
        """
        预测类别概率
        
        参数:
            X: 特征矩阵，形状为(n_samples, n_features)
            
        返回:
            类别概率，形状为(n_samples, n_classes)
        """
        if self.weights is None or self.bias is None:
            raise ValueError("模型尚未拟合，请先调用fit方法")
        
        # 计算线性输出
        logits = X @ self.weights + self.bias
        
        # 应用softmax函数获取概率
        exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        
        return probs
    
    def _to_one_hot(self, y):
        """将整数标签转换为独热编码"""
        n_samples = len(y)
        y_one_hot = np.zeros((n_samples, self.num_classes))
        
        for i, label in enumerate(y):
            label_idx = int(label) % self.num_classes  # 防止标签超出范围
            y_one_hot[i, label_idx] = 1
        
        return y_one_hot
    
    def save(self, filepath):
        """
        保存模型到文件
        
        参数:
            filepath: 保存路径
        """
        model_data = {
            'weights': self.weights,
            'bias': self.bias,
            'feature_importance': self.feature_importance,
            'num_classes': self.num_classes
        }
        
        with open(filepath, 'wb') as f:
            pickle.dump(model_data, f)
        
        self.logger.info(f"模型已保存到: {filepath}", f"Model saved to: {filepath}")
    
    def load(self, filepath):
        """
        从文件加载模型
        
        参数:
            filepath: 加载路径
        """
        with open(filepath, 'rb') as f:
            model_data = pickle.load(f)
        
        self.weights = model_data['weights']
        self.bias = model_data['bias']
        self.feature_importance = model_data['feature_importance']
        self.num_classes = model_data['num_classes']
        
        self.logger.info(f"模型已从 {filepath} 加载", f"Model loaded from: {filepath}")
        
        return self

In [ ]:
class ModelEvaluator:
    """模型评估类，用于评估模型性能并生成可视化结果"""
    
    def __init__(self, logger=None):
        """
        初始化评估器
        
        参数:
            logger: 日志记录器
        """
        self.logger = logger if logger else BilingualLogger()
        self.results = {}  # 存储评估结果
    
    def evaluate(self, model, X, y, dataset_name="test"):
        """
        评估模型性能
        
        参数:
            model: 已拟合的模型
            X: 特征矩阵
            y: 真实标签
            dataset_name: 数据集名称
            
        返回:
            包含各种评估指标的字典
        """
        self.logger.info(f"开始评估模型在{dataset_name}集上的性能", 
                       f"Starting model evaluation on {dataset_name} set")
        
        # 获取预测
        y_pred = model.predict(X)
        
        try:
            # 计算概率预测（可选）
            y_proba = model.predict_proba(X)
        except:
            y_proba = None
        
        # 计算评估指标
        accuracy = accuracy_score(y, y_pred)
        balanced_acc = balanced_accuracy_score(y, y_pred)
        
        # 处理可能的警告（某些类别可能没有样本）
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            f1_macro = f1_score(y, y_pred, average='macro')
            f1_weighted = f1_score(y, y_pred, average='weighted')
            class_f1 = f1_score(y, y_pred, average=None)
            
        # 只有当所有类别都有样本时才计算Kappa
        try:
            kappa = cohen_kappa_score(y, y_pred)
        except:
            kappa = float('nan')
        
        # 分类报告和混淆矩阵
        try:
            report = classification_report(y, y_pred, output_dict=True)
        except:
            report = {}
        
        try:
            conf_matrix = confusion_matrix(y, y_pred)
        except:
            conf_matrix = np.array([])
        
        # 统计每个类别的样本数
        class_counts = {}
        unique_classes = np.unique(y)
        for cls in unique_classes:
            class_counts[int(cls)] = np.sum(y == cls)
        
        # 将结果存储到字典中
        result = {
            'accuracy': accuracy,
            'balanced_accuracy': balanced_acc,
            'f1_macro': f1_macro,
            'f1_weighted': f1_weighted,
            'kappa': kappa,
            'class_f1': class_f1,
            'report': report,
            'confusion_matrix': conf_matrix,
            'class_counts': class_counts,
            'y_true': y,
            'y_pred': y_pred,
            'y_proba': y_proba
        }
        
        # 记录评估结果
        self.logger.info(
            f"{dataset_name}集评估结果:\n"
            f"  准确率: {accuracy:.4f}\n"
            f"  平衡准确率: {balanced_acc:.4f}\n"
            f"  宏平均F1: {f1_macro:.4f}\n"
            f"  加权F1: {f1_weighted:.4f}\n"
            f"  Kappa系数: {kappa:.4f}",
            
            f"{dataset_name} set evaluation results:\n"
            f"  Accuracy: {accuracy:.4f}\n"
            f"  Balanced Accuracy: {balanced_acc:.4f}\n"
            f"  Macro F1: {f1_macro:.4f}\n"
            f"  Weighted F1: {f1_weighted:.4f}\n"
            f"  Kappa: {kappa:.4f}"
        )
        
        # 保存结果
        self.results[dataset_name] = result
        
        return result
    
    def visualize_performance(self, save_dir=None, prefix=""):
        """
        可视化评估结果
        
        参数:
            save_dir: 图表保存目录
            prefix: 文件名前缀
        """
        if not self.results:
            self.logger.warning("没有可视化的评估结果", "No evaluation results to visualize")
            return
        
        if save_dir and not os.path.exists(save_dir):
            os.makedirs(save_dir)
        
        # 1. 绘制性能指标比较图
        self._plot_performance_comparison(save_dir, prefix)
        
        # 2. 绘制每个数据集的混淆矩阵
        for dataset_name, result in self.results.items():
            self._plot_confusion_matrix(result, dataset_name, save_dir, prefix)
        
        # 3. 绘制类别性能分析图（对测试集）
        if 'test' in self.results:
            self._plot_class_performance(self.results['test'], save_dir, prefix)
    
    def _plot_performance_comparison(self, save_dir, prefix):
        """绘制不同数据集间的性能比较"""
        # 提取数据
        datasets = list(self.results.keys())
        metrics = ['accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted', 'kappa']
        metric_names = ['Accuracy', 'Balanced Accuracy', 'Macro F1', 'Weighted F1', 'Kappa']
        
        values = np.zeros((len(metrics), len(datasets)))
        for i, dataset in enumerate(datasets):
            for j, metric in enumerate(metrics):
                values[j, i] = self.results[dataset][metric]
        
        # 绘制柱状图
        plt.figure(figsize=(12, 8))
        x = np.arange(len(metrics))
        width = 0.7 / len(datasets)
        
        for i, dataset in enumerate(datasets):
            offset = (i - len(datasets)/2 + 0.5) * width
            plt.bar(x + offset, values[:, i], width, label=dataset.capitalize())
        
        plt.xlabel('Metrics')
        plt.ylabel('Score')
        plt.title('Performance Metrics Comparison')
        plt.xticks(x, metric_names)
        plt.ylim(0, 1.0)
        plt.legend()
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        
        # 添加数值标签
        for i, dataset in enumerate(datasets):
            for j, metric in enumerate(metrics):
                offset = (i - len(datasets)/2 + 0.5) * width
                plt.text(j + offset, values[j, i] + 0.01, f"{values[j, i]:.3f}", 
                        ha='center', va='bottom', fontsize=8)
        
        plt.tight_layout()
        
        if save_dir:
            plt.savefig(os.path.join(save_dir, f"{prefix}performance_comparison.png"), dpi=300)
            plt.close()
        else:
            plt.show()
    
    def _plot_confusion_matrix(self, result, dataset_name, save_dir, prefix):
        """为指定数据集绘制混淆矩阵热图"""
        conf_matrix = result['confusion_matrix']
        
        if conf_matrix.size == 0:
            return
        
        # 对大型混淆矩阵使用对数缩放
        if conf_matrix.shape[0] > 10:
            # 添加一个小值防止log(0)
            conf_matrix_vis = np.log1p(conf_matrix)
            title = f"Confusion Matrix (log scale) - {dataset_name.capitalize()} Set"
            vmax = np.log1p(conf_matrix.max())
        else:
            conf_matrix_vis = conf_matrix
            title = f"Confusion Matrix - {dataset_name.capitalize()} Set"
            vmax = conf_matrix.max()
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(conf_matrix_vis, annot=False, fmt='d', cmap='Blues', 
                   vmin=0, vmax=vmax)
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.title(title)
        
        if save_dir:
            plt.savefig(os.path.join(save_dir, f"{prefix}confusion_matrix_{dataset_name}.png"), dpi=300)
            plt.close()
        else:
            plt.show()
    
    def _plot_class_performance(self, result, save_dir, prefix):
        """绘制类别级别的性能分析图表"""
        class_f1 = result['class_f1']
        class_counts = result['class_counts']
        
        # 准备数据
        classes = list(class_counts.keys())
        classes.sort()
        counts = [class_counts[cls] for cls in classes]
        f1_scores = [class_f1[i] if i < len(class_f1) else 0 for i in range(len(classes))]
        
        # 根据样本数量筛选类别（只显示有样本的类别）
        valid_idx = [i for i, count in enumerate(counts) if count > 0]
        classes = [classes[i] for i in valid_idx]
        counts = [counts[i] for i in valid_idx]
        f1_scores = [f1_scores[i] for i in valid_idx]
        
        if not classes:
            return
        
        # 1. 绘制F1分数柱状图（按分数排序）
        sorted_idx = np.argsort(f1_scores)
        sorted_classes = [classes[i] for i in sorted_idx]
        sorted_f1 = [f1_scores[i] for i in sorted_idx]
        
        plt.figure(figsize=(12, 6))
        plt.bar(range(len(sorted_classes)), sorted_f1)
        plt.xlabel('Class')
        plt.ylabel('F1 Score')
        plt.title('F1 Score by Class (Sorted)')
        
        # 如果类别太多，只显示一部分刻度
        if len(classes) > 20:
            tick_step = len(classes) // 10
            plt.xticks(range(0, len(sorted_classes), tick_step), 
                      [sorted_classes[i] for i in range(0, len(sorted_classes), tick_step)])
        else:
            plt.xticks(range(len(sorted_classes)), sorted_classes, rotation=90)
        
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        
        if save_dir:
            plt.savefig(os.path.join(save_dir, f"{prefix}class_f1_sorted.png"), dpi=300)
            plt.close()
        else:
            plt.show()
        
        # 2. 绘制样本数量与F1分数的散点图
        plt.figure(figsize=(10, 6))
        plt.scatter(counts, f1_scores, alpha=0.6)
        plt.xlabel('Sample Count')
        plt.ylabel('F1 Score')
        plt.title('Relationship Between Sample Count and F1 Score')
        plt.grid(True, linestyle='--', alpha=0.7)
        
        # 尝试拟合一条趋势线
        try:
            from scipy import stats
            slope, intercept, r_value, p_value, std_err = stats.linregress(counts, f1_scores)
            x = np.array([min(counts), max(counts)])
            y = slope * x + intercept
            plt.plot(x, y, 'r--', label=f'Trend line (r={r_value:.2f})')
            plt.legend()
        except:
            pass
        
        if save_dir:
            plt.savefig(os.path.join(save_dir, f"{prefix}sample_count_vs_f1.png"), dpi=300)
            plt.close()
        else:
            plt.show()
        
        # 3. 为样本最多和最少的几个类别绘制详细分析
        top_n = 5
        if len(classes) > top_n * 2:
            # 样本最多的类别
            top_idx = np.argsort(counts)[-top_n:]
            top_classes = [classes[i] for i in top_idx]
            top_counts = [counts[i] for i in top_idx]
            top_f1 = [f1_scores[i] for i in top_idx]
            
            # 样本最少的类别
            bottom_idx = np.argsort(counts)[:top_n]
            bottom_classes = [classes[i] for i in bottom_idx]
            bottom_counts = [counts[i] for i in bottom_idx]
            bottom_f1 = [f1_scores[i] for i in bottom_idx]
            
            plt.figure(figsize=(12, 6))
            
            # 左侧：样本最多的类别
            plt.subplot(1, 2, 1)
            plt.bar(range(len(top_classes)), top_f1)
            plt.title('Classes with Most Samples')
            plt.xlabel('Class')
            plt.ylabel('F1 Score')
            plt.xticks(range(len(top_classes)), top_classes, rotation=45)
            for i, (cls, count) in enumerate(zip(top_classes, top_counts)):
                plt.text(i, top_f1[i] + 0.02, f"{count}", ha='center')
            plt.grid(True, axis='y', linestyle='--', alpha=0.7)
            
            # 右侧：样本最少的类别
            plt.subplot(1, 2, 2)
            plt.bar(range(len(bottom_classes)), bottom_f1)
            plt.title('Classes with Least Samples')
            plt.xlabel('Class')
            plt.ylabel('F1 Score')
            plt.xticks(range(len(bottom_classes)), bottom_classes, rotation=45)
            for i, (cls, count) in enumerate(zip(bottom_classes, bottom_counts)):
                plt.text(i, bottom_f1[i] + 0.02, f"{count}", ha='center')
            plt.grid(True, axis='y', linestyle='--', alpha=0.7)
            
            plt.tight_layout()
            
            if save_dir:
                plt.savefig(os.path.join(save_dir, f"{prefix}top_bottom_classes.png"), dpi=300)
                plt.close()
            else:
                plt.show()

def visualize_feature_importance(model, feature_names=None, top_n=20, save_path=None):
    """
    可视化特征重要性
    
    参数:
        model: 已拟合的伪逆模型
        feature_names: 特征名称列表
        top_n: 要显示的顶部特征数量
        save_path: 图表保存路径
    """
    if model.feature_importance is None:
        print("模型尚未拟合或没有特征重要性信息")
        return
    
    # 获取特征重要性
    importance = model.feature_importance
    
    # 如果没有提供特征名称，使用索引
    if feature_names is None:
        feature_names = [f"Feature {i}" for i in range(len(importance))]
    
    # 确保特征名称长度匹配
    if len(feature_names) != len(importance):
        feature_names = [f"Feature {i}" for i in range(len(importance))]
    
    # 获取前N个重要特征
    if top_n > len(importance):
        top_n = len(importance)
    
    top_indices = np.argsort(importance)[-top_n:][::-1]
    top_importance = importance[top_indices]
    top_names = [feature_names[i] for i in top_indices]
    
    # 绘制特征重要性条形图
    plt.figure(figsize=(12, 8))
    plt.barh(range(len(top_names)), top_importance, align='center')
    plt.yticks(range(len(top_names)), top_names)
    plt.xlabel('Feature Importance (Mean Absolute Weight)')
    plt.title(f'Top {top_n} Most Important Features')
    plt.grid(True, axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

def visualize_weight_distribution(model, save_path=None):
    """
    可视化权重分布
    
    参数:
        model: 已拟合的伪逆模型
        save_path: 图表保存路径
    """
    if model.weights is None:
        print("模型尚未拟合")
        return
    
    # 将权重矩阵展平
    weights = model.weights.flatten()
    
    plt.figure(figsize=(10, 6))
    
    # 绘制权重直方图
    plt.subplot(1, 2, 1)
    plt.hist(weights, bins=50, alpha=0.7)
    plt.xlabel('Weight Value')
    plt.ylabel('Frequency')
    plt.title('Weight Distribution')
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # 绘制权重箱线图
    plt.subplot(1, 2, 2)
    plt.boxplot(weights)
    plt.ylabel('Weight Value')
    plt.title('Weight Boxplot')
    plt.grid(True, axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

In [ ]:
class ExperimentManager:
    """实验管理器，处理批量参数测试和结果管理"""
    
    def __init__(self, data_loader, base_dir="experiments", logger=None):
        """
        初始化实验管理器
        
        参数:
            data_loader: 数据加载器实例
            base_dir: 实验结果基础目录
            logger: 日志记录器
        """
        self.data_loader = data_loader
        self.base_dir = base_dir
        self.logger = logger if logger else BilingualLogger()
        
        # 确保基础目录存在
        os.makedirs(base_dir, exist_ok=True)
        
        # 创建实验记录文件
        self.experiment_log_path = os.path.join(base_dir, "experiment_log.csv")
        if not os.path.exists(self.experiment_log_path):
            with open(self.experiment_log_path, 'w') as f:
                f.write("experiment_id,timestamp,apply_pca,n_components,normalization,"
                       "class_balance,target_samples,regularization,alpha,"
                       "train_accuracy,test_accuracy,val_accuracy,train_f1,test_f1,val_f1,"
                       "train_time,evaluation_time,status\n")
        
        self.logger.info(f"实验管理器初始化，基础目录: {base_dir}", 
                       f"Experiment manager initialized with base directory: {base_dir}")
    
    # def run_experiment(self, params, force_rerun=False):
    #     """
    #     运行单个实验
        
    #     参数:
    #         params: 实验参数字典
    #         force_rerun: 是否强制重新运行已完成的实验
            
    #     返回:
    #         experiment_id: 实验ID
    #         results: 实验结果
    #     """
    #     # 生成实验ID
    #     experiment_id = self._generate_experiment_id(params)
    #     experiment_dir = os.path.join(self.base_dir, experiment_id)
        
    #     # 检查是否已完成实验
    #     if os.path.exists(experiment_dir) and not force_rerun:
    #         status_path = os.path.join(experiment_dir, "status.json")
    #         if os.path.exists(status_path):
    #             with open(status_path, 'r') as f:
    #                 status = json.load(f)
    #                 if status.get('status') == 'completed':
    #                     self.logger.info(f"实验 {experiment_id} 已完成，跳过", 
    #                                    f"Experiment {experiment_id} already completed, skipping")
    #                     return experiment_id, self._load_results(experiment_dir)
        
    #     # 创建实验目录
    #     os.makedirs(experiment_dir, exist_ok=True)
        
    #     # 保存参数
    #     with open(os.path.join(experiment_dir, "params.json"), 'w') as f:
    #         json.dump(params, f, indent=4)
        
    #     # 更新状态为运行中
    #     with open(os.path.join(experiment_dir, "status.json"), 'w') as f:
    #         json.dump({"status": "running", "start_time": str(datetime.now())}, f, indent=4)
        
    #     # 记录实验开始
    #     self.logger.info(f"开始实验 {experiment_id}", f"Starting experiment {experiment_id}")
    #     self.logger.info(f"参数: {params}", f"Parameters: {params}")
        
    #     try:
    #         # 处理数据
    #         start_time = time.time()
    #         processed_data = self.data_loader.preprocess_data(
    #             apply_pca=params.get('apply_pca', False),
    #             n_components=params.get('n_components', 50),
    #             normalization=params.get('normalization', None),
    #             class_balance=params.get('class_balance', False),
    #             target_samples=params.get('target_samples', 1000)
    #         )
            
    #         # 训练模型
    #         model = PseudoInverseModel(num_classes=102, logger=self.logger)
    #         model.fit(
    #             processed_data['train_X'], 
    #             processed_data['train_y'],
    #             regularization=params.get('regularization', None),
    #             alpha=params.get('alpha', 0.0)
    #         )
    #         train_time = time.time() - start_time
            
    #         # 保存模型
    #         model.save(os.path.join(experiment_dir, "model.pkl"))
            
    #         # 评估模型
    #         eval_start_time = time.time()
    #         evaluator = ModelEvaluator(logger=self.logger)
            
    #         # 评估训练集
    #         train_result = evaluator.evaluate(
    #             model, processed_data['train_X'], processed_data['train_y'], "train")
            
    #         # 评估测试集
    #         test_result = evaluator.evaluate(
    #             model, processed_data['test_X'], processed_data['test_y'], "test")
            
    #         # 评估验证集
    #         val_result = evaluator.evaluate(
    #             model, processed_data['val_X'], processed_data['val_y'], "val")
            
    #         evaluation_time = time.time() - eval_start_time
            
    #         # 生成可视化
    #         evaluator.visualize_performance(save_dir=experiment_dir)
            
    #         # 特征重要性可视化
    #         visualize_feature_importance(
    #             model, top_n=30, 
    #             save_path=os.path.join(experiment_dir, "feature_importance.png"))
            
    #         # 权重分布可视化
    #         visualize_weight_distribution(
    #             model, save_path=os.path.join(experiment_dir, "weight_distribution.png"))
            
    #         # 保存评估结果
    #         results = {
    #             'train': train_result,
    #             'test': test_result,
    #             'val': val_result,
    #             'train_time': train_time,
    #             'evaluation_time': evaluation_time
    #         }

    #         # 保存评估结果
    #         with open(os.path.join(experiment_dir, "results.json"), 'w') as f:
    #             # 转换numpy数组为列表以便JSON序列化
    #             results_json = {}
    #             for dataset, result in results.items():
    #                 if isinstance(result, dict):
    #                     results_json[dataset] = {}
    #                     for k, v in result.items():
    #                         if isinstance(v, np.ndarray):
    #                             results_json[dataset][k] = v.tolist() if v.size > 0 else []
    #                         elif k == 'report' and isinstance(v, dict):
    #                             results_json[dataset][k] = v
    #                         elif not isinstance(v, (dict, list, str, int, float, bool, type(None))):
    #                             results_json[dataset][k] = str(v)
    #                         else:
    #                             results_json[dataset][k] = v
    #                 else:
    #                     results_json[dataset] = result
                
    #             json.dump(results_json, f, indent=4)

    #         # 更新状态为已完成
    #         with open(os.path.join(experiment_dir, "status.json"), 'w') as f:
    #             json.dump({
    #                 "status": "completed", 
    #                 "start_time": str(datetime.now()),
    #                 "end_time": str(datetime.now()),
    #                 "train_time": train_time,
    #                 "evaluation_time": evaluation_time
    #             }, f, indent=4)
            
    #         # 更新实验日志
    #         self._update_experiment_log(
    #             experiment_id, params, train_result, test_result, val_result, 
    #             train_time, evaluation_time, "completed")
            
    #         self.logger.info(f"实验 {experiment_id} 完成", f"Experiment {experiment_id} completed")
            
    #         return experiment_id, results
            
    #     except Exception as e:
    #         # 记录错误
    #         self.logger.error(f"实验 {experiment_id} 失败: {str(e)}", 
    #                         f"Experiment {experiment_id} failed: {str(e)}")
            
    #         # 更新状态为失败
    #         with open(os.path.join(experiment_dir, "status.json"), 'w') as f:
    #             json.dump({
    #                 "status": "failed",
    #                 "error": str(e),
    #                 "traceback": traceback.format_exc()
    #             }, f, indent=4)
            
    #         # 更新实验日志
    #         self._update_experiment_log(
    #             experiment_id, params, None, None, None, 0, 0, "failed")
            
    #         raise e
    
    def run_batch_experiments(self, param_grid, max_experiments=None):
        """
        运行批量实验
        
        参数:
            param_grid: 参数网格，包含每个参数的可能值列表
            max_experiments: 最大实验数量，None表示不限制
            
        返回:
            completed_experiments: 已完成实验的ID列表
        """
        # 生成所有参数组合
        param_keys = list(param_grid.keys())
        param_values = list(param_grid.values())
        
        # 计算总实验数
        total_combinations = 1
        for values in param_values:
            total_combinations *= len(values)
        
        if max_experiments and max_experiments < total_combinations:
            self.logger.info(f"限制实验数量为 {max_experiments}/{total_combinations}", 
                           f"Limiting to {max_experiments}/{total_combinations} experiments")
            total_combinations = max_experiments
        
        self.logger.info(f"开始批量实验，共 {total_combinations} 个组合", 
                       f"Starting batch experiments with {total_combinations} combinations")
        
        # 生成所有参数组合
        param_combinations = []
        
        def generate_combinations(keys, values, current=0, current_params={}):
            if current == len(keys):
                param_combinations.append(current_params.copy())
                return
            
            for value in values[current]:
                current_params[keys[current]] = value
                generate_combinations(keys, values, current + 1, current_params)
                
                # 如果达到最大实验数，则停止
                if max_experiments and len(param_combinations) >= max_experiments:
                    break
        
        generate_combinations(param_keys, param_values)
        
        # 打乱参数组合，使实验更加随机化
        np.random.shuffle(param_combinations)
        
        # 运行所有实验
        completed_experiments = []
        total_start_time = time.time()
        
        for i, params in enumerate(param_combinations):
            # 估计剩余时间
            if i > 0:
                elapsed_time = time.time() - total_start_time
                avg_time_per_exp = elapsed_time / i
                remaining_time = avg_time_per_exp * (len(param_combinations) - i)
                
                self.logger.info(
                    f"进度: {i}/{len(param_combinations)} ({i/len(param_combinations)*100:.1f}%), "
                    f"预计剩余时间: {remaining_time/60:.1f} 分钟",
                    
                    f"Progress: {i}/{len(param_combinations)} ({i/len(param_combinations)*100:.1f}%), "
                    f"Estimated time remaining: {remaining_time/60:.1f} minutes"
                )
            
            try:
                experiment_id, _ = self.run_experiment(params)
                completed_experiments.append(experiment_id)
            except Exception as e:
                self.logger.error(f"实验失败: {str(e)}", f"Experiment failed: {str(e)}")
                # 继续下一个实验
                continue
        
        total_elapsed_time = time.time() - total_start_time
        self.logger.info(
            f"批量实验完成，共 {len(completed_experiments)}/{len(param_combinations)} 个实验成功，"
            f"总耗时: {total_elapsed_time/60:.1f} 分钟",
            
            f"Batch experiments completed, {len(completed_experiments)}/{len(param_combinations)} "
            f"experiments succeeded, total time: {total_elapsed_time/60:.1f} minutes"
        )
        
        return completed_experiments
    
    # def generate_summary_report(self, top_n=10):
    #     """
    #     生成汇总报告
        
    #     参数:
    #         top_n: 展示的顶部实验数量
    #     """
    #     # 读取实验日志
    #     if not os.path.exists(self.experiment_log_path):
    #         self.logger.warning("实验日志不存在", "Experiment log does not exist")
    #         return
        
    #     try:
    #         log_df = pd.read_csv(self.experiment_log_path)
    #     except:
    #         self.logger.warning("无法读取实验日志", "Cannot read experiment log")
    #         return
        
    #     # 过滤成功的实验
    #     log_df = log_df[log_df['status'] == 'completed']
        
    #     if len(log_df) == 0:
    #         self.logger.warning("没有已完成的实验", "No completed experiments")
    #         return
        
    #     # 按测试集准确率排序
    #     log_df_sorted = log_df.sort_values('test_accuracy', ascending=False)
        
    #     # 创建报告目录
    #     report_dir = os.path.join(self.base_dir, "summary_report")
    #     os.makedirs(report_dir, exist_ok=True)
        
    #     # 保存排序后的实验日志
    #     log_df_sorted.to_csv(os.path.join(report_dir, "experiments_sorted.csv"), index=False)
        
    #     # 生成顶部实验表格
    #     top_df = log_df_sorted.head(top_n)
        
    #     # 生成HTML报告
    #     html_report = f"""
    #     <html>
    #     <head>
    #         <title>Pseudo-Inverse Experiments Summary Report</title>
    #         <style>
    #             body {{ font-family: Arial, sans-serif; margin: 20px; }}
    #             h1 {{ color: #333366; }}
    #             h2 {{ color: #666699; }}
    #             table {{ border-collapse: collapse; width: 100%; }}
    #             th, td {{ padding: 8px; text-align: left; border-bottom: 1px solid #ddd; }}
    #             th {{ background-color: #f2f2f2; }}
    #             tr:hover {{ background-color: #f5f5f5; }}
    #             .chart-container {{ display: flex; flex-wrap: wrap; justify-content: space-between; }}
    #             .chart {{ margin: 10px; max-width: 600px; }}
    #         </style>
    #     </head>
    #     <body>
    #         <h1>Pseudo-Inverse Experiments Summary Report</h1>
    #         <p>Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    #         <p>Total experiments: {len(log_df)}</p>
            
    #         <h2>Top {top_n} Experiments by Test Accuracy</h2>
    #         <table>
    #             <tr>
    #                 <th>Experiment ID</th>
    #                 <th>Test Accuracy</th>
    #                 <th>Test F1</th>
    #                 <th>Val Accuracy</th>
    #                 <th>PCA</th>
    #                 <th>Components</th>
    #                 <th>Normalization</th>
    #                 <th>Class Balance</th>
    #                 <th>Regularization</th>
    #             </tr>
    #     """
        
    #     for _, row in top_df.iterrows():
    #         html_report += f"""
    #             <tr>
    #                 <td>{row['experiment_id']}</td>
    #                 <td>{row['test_accuracy']:.4f}</td>
    #                 <td>{row['test_f1']:.4f}</td>
    #                 <td>{row['val_accuracy']:.4f}</td>
    #                 <td>{'Yes' if row['apply_pca'] else 'No'}</td>
    #                 <td>{row['n_components']}</td>
    #                 <td>{row['normalization']}</td>
    #                 <td>{'Yes' if row['class_balance'] else 'No'}</td>
    #                 <td>{row['regularization']}</td>
    #             </tr>
    #         """
        
    #     html_report += """
    #         </table>
            
    #         <h2>Parameter Impact Analysis</h2>
    #         <div class="chart-container">
    #     """
        
    #     # 生成参数影响可视化
    #     # 1. PCA对准确率的影响
    #     if 'apply_pca' in log_df.columns:
    #         plt.figure(figsize=(8, 6))
    #         sns.boxplot(x='apply_pca', y='test_accuracy', data=log_df)
    #         plt.title('Impact of PCA on Test Accuracy')
    #         plt.xlabel('PCA Applied')
    #         plt.ylabel('Test Accuracy')
    #         plt.tight_layout()
    #         pca_impact_path = os.path.join(report_dir, "pca_impact.png")
    #         plt.savefig(pca_impact_path, dpi=300)
    #         plt.close()
            
    #         html_report += f"""
    #             <div class="chart">
    #                 <img src="{os.path.relpath(pca_impact_path, self.base_dir)}" alt="PCA Impact" width="100%">
    #             </div>
    #         """
        
    #     # 2. 组件数量对准确率的影响
    #     if 'n_components' in log_df.columns and 'apply_pca' in log_df.columns:
    #         pca_df = log_df[log_df['apply_pca'] == True]
    #         if len(pca_df) > 1:
    #             plt.figure(figsize=(8, 6))
    #             sns.scatterplot(x='n_components', y='test_accuracy', data=pca_df)
    #             plt.title('Impact of PCA Components on Test Accuracy')
    #             plt.xlabel('Number of Components')
    #             plt.ylabel('Test Accuracy')
    #             plt.grid(True, linestyle='--', alpha=0.7)
    #             plt.tight_layout()
    #             components_impact_path = os.path.join(report_dir, "components_impact.png")
    #             plt.savefig(components_impact_path, dpi=300)
    #             plt.close()
                
    #             html_report += f"""
    #                 <div class="chart">
    #                     <img src="{os.path.relpath(components_impact_path, self.base_dir)}" alt="Components Impact" width="100%">
    #                 </div>
    #             """
        
    #     # 3. 标准化方法对准确率的影响
    #     if 'normalization' in log_df.columns:
    #         plt.figure(figsize=(8, 6))
    #         sns.boxplot(x='normalization', y='test_accuracy', data=log_df)
    #         plt.title('Impact of Normalization on Test Accuracy')
    #         plt.xlabel('Normalization Method')
    #         plt.ylabel('Test Accuracy')
    #         plt.tight_layout()
    #         norm_impact_path = os.path.join(report_dir, "normalization_impact.png")
    #         plt.savefig(norm_impact_path, dpi=300)
    #         plt.close()
            
    #         html_report += f"""
    #             <div class="chart">
    #                 <img src="{os.path.relpath(norm_impact_path, self.base_dir)}" alt="Normalization Impact" width="100%">
    #             </div>
    #         """
        
    #     # 4. 类别平衡对准确率的影响
    #     if 'class_balance' in log_df.columns:
    #         plt.figure(figsize=(8, 6))
    #         sns.boxplot(x='class_balance', y='test_accuracy', data=log_df)
    #         plt.title('Impact of Class Balancing on Test Accuracy')
    #         plt.xlabel('Class Balancing Applied')
    #         plt.ylabel('Test Accuracy')
    #         plt.tight_layout()
    #         balance_impact_path = os.path.join(report_dir, "class_balance_impact.png")
    #         plt.savefig(balance_impact_path, dpi=300)
    #         plt.close()
            
    #         html_report += f"""
    #             <div class="chart">
    #                 <img src="{os.path.relpath(balance_impact_path, self.base_dir)}" alt="Class Balance Impact" width="100%">
    #             </div>
    #         """
        
    #     # 5. 正则化方法对准确率的影响
    #     if 'regularization' in log_df.columns:
    #         plt.figure(figsize=(8, 6))
    #         sns.boxplot(x='regularization', y='test_accuracy', data=log_df)
    #         plt.title('Impact of Regularization on Test Accuracy')
    #         plt.xlabel('Regularization Method')
    #         plt.ylabel('Test Accuracy')
    #         plt.tight_layout()
    #         reg_impact_path = os.path.join(report_dir, "regularization_impact.png")
    #         plt.savefig(reg_impact_path, dpi=300)
    #         plt.close()
            
    #         html_report += f"""
    #             <div class="chart">
    #                 <img src="{os.path.relpath(reg_impact_path, self.base_dir)}" alt="Regularization Impact" width="100%">
    #             </div>
    #         """
        
    #     html_report += """
    #         </div>
            
    #         <h2>Best Experiment Details</h2>
    #     """
        
    #     # 获取最佳实验的详细信息
    #     if len(top_df) > 0:
    #         best_exp_id = top_df.iloc[0]['experiment_id']
    #         best_exp_dir = os.path.join(self.base_dir, best_exp_id)
            
    #         # 复制最佳实验的图表到报告目录
    #         for img_file in ['feature_importance.png', 'weight_distribution.png', 
    #                         'performance_comparison.png', 'confusion_matrix_test.png']:
    #             img_path = os.path.join(best_exp_dir, img_file)
    #             if os.path.exists(img_path):
    #                 dest_path = os.path.join(report_dir, f"best_{img_file}")
    #                 import shutil
    #                 shutil.copy(img_path, dest_path)
                    
    #                 html_report += f"""
    #                     <div class="chart">
    #                         <h3>{img_file.replace('_', ' ').replace('.png', '').title()}</h3>
    #                         <img src="{os.path.relpath(dest_path, self.base_dir)}" alt="{img_file}" width="100%">
    #                     </div>
    #                 """
        
    #     html_report += """
    #         <h2>Conclusion and Recommendations</h2>
    #         <p>Based on the experimental results, here are the key findings:</p>
    #         <ul>
    #     """
        
    #     # 添加结论（基于实验结果）
    #     # PCA
    #     if 'apply_pca' in log_df.columns:
    #         pca_impact = log_df.groupby('apply_pca')['test_accuracy'].mean()
    #         if len(pca_impact) > 1:
    #             pca_better = pca_impact[True] > pca_impact[False]
    #             html_report += f"""
    #                 <li>PCA {'improves' if pca_better else 'reduces'} model performance: 
    #                     Average accuracy with PCA: {pca_impact[True]:.4f}, 
    #                     without PCA: {pca_impact[False]:.4f}</li>
    #             """
        
    #     # 标准化
    #     if 'normalization' in log_df.columns:
    #         norm_impact = log_df.groupby('normalization')['test_accuracy'].mean()
    #         if len(norm_impact) > 1:
    #             best_norm = norm_impact.idxmax()
    #             html_report += f"""
    #                 <li>Best normalization method: {best_norm} (Avg. accuracy: {norm_impact[best_norm]:.4f})</li>
    #             """
        
    #     # 类别平衡
    #     if 'class_balance' in log_df.columns:
    #         balance_impact = log_df.groupby('class_balance')['test_accuracy'].mean()
    #         if len(balance_impact) > 1:
    #             balance_better = balance_impact[True] > balance_impact[False]
    #             html_report += f"""
    #                 <li>Class balancing {'improves' if balance_better else 'reduces'} model performance: 
    #                     Average accuracy with balancing: {balance_impact[True]:.4f}, 
    #                     without balancing: {balance_impact[False]:.4f}</li>
    #             """
        
    #     # 正则化
    #     if 'regularization' in log_df.columns:
    #         reg_impact = log_df.groupby('regularization')['test_accuracy'].mean()
    #         if len(reg_impact) > 1:
    #             best_reg = reg_impact.idxmax()
    #             html_report += f"""
    #                 <li>Best regularization method: {best_reg} (Avg. accuracy: {reg_impact[best_reg]:.4f})</li>
    #             """
        
    #     # 最佳参数组合
    #     if len(top_df) > 0:
    #         best_row = top_df.iloc[0]
    #         html_report += f"""
    #             <li>Best parameter combination:
    #                 <ul>
    #                     <li>PCA: {'Yes' if best_row['apply_pca'] else 'No'}</li>
    #                     {'<li>Components: ' + str(best_row['n_components']) + '</li>' if best_row['apply_pca'] else ''}
    #                     <li>Normalization: {best_row['normalization']}</li>
    #                     <li>Class balancing: {'Yes' if best_row['class_balance'] else 'No'}</li>
    #                     <li>Regularization: {best_row['regularization']}</li>
    #                     <li>Alpha: {best_row['alpha']}</li>
    #                 </ul>
    #             </li>
    #         """
        
    #     html_report += """
    #         </ul>
    #     </body>
    #     </html>
    #     """
        
    #     # 保存HTML报告
    #     with open(os.path.join(report_dir, "summary_report.html"), 'w') as f:
    #         f.write(html_report)
        
    #     self.logger.info(f"汇总报告已生成: {os.path.join(report_dir, 'summary_report.html')}", 
    #                    f"Summary report generated: {os.path.join(report_dir, 'summary_report.html')}")
    
    def _generate_experiment_id(self, params):
        """生成唯一的实验ID"""
        # 使用参数的哈希值作为ID的一部分
        param_str = '_'.join([f"{k}_{v}" for k, v in sorted(params.items())])
        param_hash = abs(hash(param_str)) % 10000
        
        # 添加时间戳
        timestamp = datetime.now().strftime("%m%d_%H%M%S")
        
        # 生成ID
        experiment_id = f"exp_{timestamp}_{param_hash}"
        
        return experiment_id
    
    # def _update_experiment_log(self, experiment_id, params, train_result, test_result, 
    #                          val_result, train_time, evaluation_time, status):
    #     """更新实验日志"""
    #     # 提取评估指标
    #     train_acc = train_result['accuracy'] if train_result else float('nan')
    #     test_acc = test_result['accuracy'] if test_result else float('nan')
    #     val_acc = val_result['accuracy'] if val_result else float('nan')
        
    #     train_f1 = train_result['f1_macro'] if train_result else float('nan')
    #     test_f1 = test_result['f1_macro'] if test_result else float('nan')
    #     val_f1 = val_result['f1_macro'] if val_result else float('nan')
        
    #     # 准备日志条目
    #     log_entry = (f"{experiment_id},{datetime.now().strftime('%Y-%m-%d %H:%M:%S')},"
    #                 f"{params.get('apply_pca', False)},{params.get('n_components', 0)},"
    #                 f"{params.get('normalization', 'none')},{params.get('class_balance', False)},"
    #                 f"{params.get('target_samples', 0)},{params.get('regularization', 'none')},"
    #                 f"{params.get('alpha', 0.0)},"
    #                 f"{train_acc},{test_acc},{val_acc},{train_f1},{test_f1},{val_f1},"
    #                 f"{train_time},{evaluation_time},{status}\n")
        
    #     # 追加到日志文件
    #     with open(self.experiment_log_path, 'a') as f:
    #         f.write(log_entry)
    
    def _load_results(self, experiment_dir):
        """加载保存的实验结果"""
        results_path = os.path.join(experiment_dir, "results.json")
        if os.path.exists(results_path):
            try:
                with open(results_path, 'r') as f:
                    return json.load(f)
            except:
                return None
        return None

In [ ]:
# 扩展实验管理器以支持特征选择
class ExperimentManagerWithFeatureSelection(ExperimentManager):
    """扩展实验管理器，支持特征选择"""
    
    def __init__(self, data_loader, base_dir="experiments", logger=None, 
                feature_selection_mode='global'):
        """
        初始化实验管理器
        
        参数:
            data_loader: 数据加载器实例
            base_dir: 实验结果基础目录
            logger: 日志记录器
            feature_selection_mode: 特征选择模式，'global'或'per_experiment'
        """
        super().__init__(data_loader, base_dir, logger)
        self.feature_selection_mode = feature_selection_mode
        
        # 全局特征选择器
        self.global_selector = None
        # 全局特征选择前后的原始数据和处理后数据
        self.original_data = None
        self.global_selected_data = None
        
        # 更新实验日志文件头
        self.experiment_log_path = os.path.join(base_dir, "experiment_log_with_fs.csv")
        if not os.path.exists(self.experiment_log_path):
            with open(self.experiment_log_path, 'w') as f:
                f.write("experiment_id,timestamp,apply_pca,n_components,normalization,"
                       "class_balance,target_samples,regularization,alpha,"
                       "feature_selection,selection_mode,selection_threshold,max_features,l1_ratio,"
                       "train_accuracy,test_accuracy,val_accuracy,train_f1,test_f1,val_f1,"
                       "original_features,selected_features,selection_ratio,"
                       "train_time,evaluation_time,status\n")
        
        self.logger.info(f"扩展实验管理器初始化，特征选择模式: {feature_selection_mode}", 
                       f"Extended experiment manager initialized with feature selection mode: {feature_selection_mode}")
    
    def apply_global_feature_selection(self, selection_params):
        """
        应用全局特征选择
        
        参数:
            selection_params: 特征选择参数字典
            
        返回:
            feature_selector: 训练好的特征选择器
        """
        self.logger.info("开始全局特征选择", "Starting global feature selection")
        
        # 加载原始数据
        if self.original_data is None:
            self.original_data = self.data_loader.load_all_data()
        
        # 应用预处理，但不包括特征选择
        preprocessed_data = self.data_loader.preprocess_data(
            apply_pca=selection_params.get('apply_pca', False),
            n_components=selection_params.get('n_components', 50),
            normalization=selection_params.get('normalization', 'standard'),
            class_balance=selection_params.get('class_balance', False),
            target_samples=selection_params.get('target_samples', 1000)
        )
        
        # 创建特征选择器
        self.global_selector = FeatureSelector(
            method=selection_params.get('feature_selection', 'lasso'),
            selection_mode=selection_params.get('selection_mode', 'threshold'),
            selection_threshold=selection_params.get('selection_threshold', 0.01),
            max_features=selection_params.get('max_features', 100),
            l1_ratio=selection_params.get('l1_ratio', 1.0),
            cv_folds=selection_params.get('cv_folds', 5),
            random_state=42,
            scaling_before_selection=selection_params.get('scaling_before_selection', True),
            selection_metric=selection_params.get('selection_metric', 'coefficient'),
            logger=self.logger
        )
        
        # 拟合特征选择器
        self.global_selector.fit(
            preprocessed_data['train_X'], 
            preprocessed_data['train_y']
        )
        
        # 转换所有数据集
        train_X_selected = self.global_selector.transform(preprocessed_data['train_X'])
        test_X_selected = self.global_selector.transform(preprocessed_data['test_X'])
        val_X_selected = self.global_selector.transform(preprocessed_data['val_X'])
        
        # 保存全局特征选择后的数据
        self.global_selected_data = {
            'train_X': train_X_selected,
            'train_y': preprocessed_data['train_y'],
            'test_X': test_X_selected,
            'test_y': preprocessed_data['test_y'],
            'val_X': val_X_selected,
            'val_y': preprocessed_data['val_y'],
            'original_dim': preprocessed_data['train_X'].shape[1],
            'selected_dim': train_X_selected.shape[1]
        }
        
        
        # 评估特征选择的稳定性
        stability_metrics = self.global_selector.evaluate_stability(
            preprocessed_data['train_X'], 
            preprocessed_data['train_y']
        )
        
        # 将稳定性指标保存到全局选择器中，以便后续可视化使用
        self.global_selector.selection_frequency = stability_metrics['selection_frequency']
        self.global_selector.jaccard_matrix = stability_metrics['jaccard_matrix']

        # 保存全局特征选择结果
        global_fs_dir = os.path.join(self.base_dir, "global_feature_selection")
        os.makedirs(global_fs_dir, exist_ok=True)
        
        # 保存特征选择器
        self.global_selector.save(os.path.join(global_fs_dir, "global_selector.pkl"))
        
        # 可视化特征选择结果
        visualize_feature_selection(
            self.global_selector, 
            save_path=os.path.join(global_fs_dir, "feature_selection_visualization.png")
        )
        
        # 可视化特征选择稳定性
        visualize_feature_stability(
            stability_metrics, 
            save_path=os.path.join(global_fs_dir, "feature_stability_visualization.png")
        )
        
        self.logger.info(f"全局特征选择完成，从 {preprocessed_data['train_X'].shape[1]} 个特征中选择了 "
                      f"{train_X_selected.shape[1]} 个特征",
                      f"Global feature selection completed, selected {train_X_selected.shape[1]} "
                      f"features from {preprocessed_data['train_X'].shape[1]}")
        
        return self.global_selector
    
    def run_experiment(self, params, force_rerun=False):
        """
        运行单个实验，支持特征选择
        
        参数:
            params: 实验参数字典
            force_rerun: 是否强制重新运行已完成的实验
            
        返回:
            experiment_id: 实验ID
            results: 实验结果
        """
        # 生成实验ID
        experiment_id = self._generate_experiment_id(params)
        experiment_dir = os.path.join(self.base_dir, experiment_id)
        
        # 检查是否已完成实验
        if os.path.exists(experiment_dir) and not force_rerun:
            status_path = os.path.join(experiment_dir, "status.json")
            if os.path.exists(status_path):
                with open(status_path, 'r') as f:
                    status = json.load(f)
                    if status.get('status') == 'completed':
                        self.logger.info(f"实验 {experiment_id} 已完成，跳过", 
                                       f"Experiment {experiment_id} already completed, skipping")
                        return experiment_id, self._load_results(experiment_dir)
        
        # 创建实验目录
        os.makedirs(experiment_dir, exist_ok=True)
        
        # 保存参数
        with open(os.path.join(experiment_dir, "params.json"), 'w') as f:
            json.dump(params, f, indent=4)
        
        # 更新状态为运行中
        with open(os.path.join(experiment_dir, "status.json"), 'w') as f:
            json.dump({"status": "running", "start_time": str(datetime.now())}, f, indent=4)
        
        # 记录实验开始
        self.logger.info(f"开始实验 {experiment_id}", f"Starting experiment {experiment_id}")
        self.logger.info(f"参数: {params}", f"Parameters: {params}")
        
        try:
            # 判断是否使用特征选择
            use_feature_selection = params.get('feature_selection') is not None
            
            # 处理数据
            start_time = time.time()
            
            # 根据特征选择模式处理数据
            if use_feature_selection and self.feature_selection_mode == 'global':
                # 全局特征选择模式
                if self.global_selector is None:
                    # 如果全局选择器尚未创建，则创建
                    self.apply_global_feature_selection(params)
                
                # 使用全局选择后的数据
                processed_data = {
                    'train_X': self.global_selected_data['train_X'],
                    'train_y': self.global_selected_data['train_y'],
                    'test_X': self.global_selected_data['test_X'],
                    'test_y': self.global_selected_data['test_y'],
                    'val_X': self.global_selected_data['val_X'],
                    'val_y': self.global_selected_data['val_y']
                }
                
                # 记录原始和选择后的特征维度
                original_dim = self.global_selected_data['original_dim']
                selected_dim = self.global_selected_data['selected_dim']
                
            elif use_feature_selection and self.feature_selection_mode == 'per_experiment':
                # 每次实验单独进行特征选择
                processed_data = self.data_loader.preprocess_data_with_feature_selection(
                    apply_pca=params.get('apply_pca', False),
                    n_components=params.get('n_components', 50),
                    normalization=params.get('normalization', None),
                    class_balance=params.get('class_balance', False),
                    target_samples=params.get('target_samples', 1000),
                    feature_selection=params.get('feature_selection'),
                    selection_mode=params.get('selection_mode', 'threshold'),
                    selection_threshold=params.get('selection_threshold', 0.01),
                    max_features=params.get('max_features', 100),
                    l1_ratio=params.get('l1_ratio', 1.0),
                    cv_folds=params.get('cv_folds', 5),
                    scaling_before_selection=params.get('scaling_before_selection', True),
                    selection_metric=params.get('selection_metric', 'coefficient')
                )
                
                # 获取特征选择器并保存
                selector = processed_data.pop('feature_selector')
                selector.save(os.path.join(experiment_dir, "feature_selector.pkl"))
                
                # 可视化特征选择结果
                visualize_feature_selection(
                    selector, 
                    save_path=os.path.join(experiment_dir, "feature_selection_visualization.png")
                )
                
                # 记录原始和选择后的特征维度
                original_dim = selector.feature_importance.shape[0]
                selected_dim = len(selector.selected_indices)
                
            else:
                # 不使用特征选择，正常处理数据
                processed_data = self.data_loader.preprocess_data(
                    apply_pca=params.get('apply_pca', False),
                    n_components=params.get('n_components', 50),
                    normalization=params.get('normalization', None),
                    class_balance=params.get('class_balance', False),
                    target_samples=params.get('target_samples', 1000)
                )
                
                # 原始和选择后的特征维度相同
                original_dim = processed_data['train_X'].shape[1]
                selected_dim = original_dim
            
            # 训练伪逆模型
            model = PseudoInverseModel(num_classes=102, logger=self.logger)
            model.fit(
                processed_data['train_X'], 
                processed_data['train_y'],
                regularization=params.get('regularization', None),
                alpha=params.get('alpha', 0.0)
            )
            
            train_time = time.time() - start_time
            
            # 保存模型
            model.save(os.path.join(experiment_dir, "model.pkl"))
            
            # 评估模型
            eval_start_time = time.time()
            evaluator = ModelEvaluator(logger=self.logger)
            
            # 评估训练集
            train_result = evaluator.evaluate(
                model, processed_data['train_X'], processed_data['train_y'], "train")
            
            # 评估测试集
            test_result = evaluator.evaluate(
                model, processed_data['test_X'], processed_data['test_y'], "test")
            
            # 评估验证集
            val_result = evaluator.evaluate(
                model, processed_data['val_X'], processed_data['val_y'], "val")
            
            evaluation_time = time.time() - eval_start_time
            
            # 生成可视化
            evaluator.visualize_performance(save_dir=experiment_dir)
            
            # 特征重要性可视化
            visualize_feature_importance(
                model, top_n=30, 
                save_path=os.path.join(experiment_dir, "feature_importance.png"))
            
            # 权重分布可视化
            visualize_weight_distribution(
                model, save_path=os.path.join(experiment_dir, "weight_distribution.png"))
            
            # 保存评估结果
            results = {
                'train': train_result,
                'test': test_result,
                'val': val_result,
                'train_time': train_time,
                'evaluation_time': evaluation_time,
                'original_dim': original_dim,
                'selected_dim': selected_dim
            }
            
            # 保存评估结果
            with open(os.path.join(experiment_dir, "results.json"), 'w') as f:
                # 转换numpy数组为列表以便JSON序列化
                results_json = {}
                for dataset, result in results.items():
                    if isinstance(result, dict):
                        results_json[dataset] = {}
                        for k, v in result.items():
                            if isinstance(v, np.ndarray):
                                results_json[dataset][k] = v.tolist() if v.size > 0 else []
                            elif k == 'report' and isinstance(v, dict):
                                results_json[dataset][k] = v
                            elif not isinstance(v, (dict, list, str, int, float, bool, type(None))):
                                results_json[dataset][k] = str(v)
                            else:
                                results_json[dataset][k] = v
                    else:
                        results_json[dataset] = result
                
                json.dump(results_json, f, indent=4)
                
            # 更新状态为已完成
            with open(os.path.join(experiment_dir, "status.json"), 'w') as f:
                json.dump({
                    "status": "completed", 
                    "start_time": str(datetime.now()),
                    "end_time": str(datetime.now()),
                    "train_time": train_time,
                    "evaluation_time": evaluation_time,
                    "original_dim": original_dim,
                    "selected_dim": selected_dim
                }, f, indent=4)
            
            # 更新实验日志
            self._update_experiment_log_with_fs(
                experiment_id, params, train_result, test_result, val_result, 
                original_dim, selected_dim, train_time, evaluation_time, "completed")
            
            self.logger.info(f"实验 {experiment_id} 完成", f"Experiment {experiment_id} completed")
            
            return experiment_id, results
            
        except Exception as e:
            # 记录错误
            self.logger.error(f"实验 {experiment_id} 失败: {str(e)}", 
                            f"Experiment {experiment_id} failed: {str(e)}")
            
            # 更新状态为失败
            with open(os.path.join(experiment_dir, "status.json"), 'w') as f:
                json.dump({
                    "status": "failed",
                    "error": str(e),
                    "traceback": traceback.format_exc()
                }, f, indent=4)
            
            # 更新实验日志
            self._update_experiment_log_with_fs(
                experiment_id, params, None, None, None, 0, 0, 0, 0, "failed")
            
            raise e
    
    def _update_experiment_log_with_fs(self, experiment_id, params, train_result, test_result, 
                                    val_result, original_dim, selected_dim, train_time, 
                                    evaluation_time, status):
        """更新包含特征选择信息的实验日志"""
        # 提取评估指标
        train_acc = train_result['accuracy'] if train_result else float('nan')
        test_acc = test_result['accuracy'] if test_result else float('nan')
        val_acc = val_result['accuracy'] if val_result else float('nan')
        
        train_f1 = train_result['f1_macro'] if train_result else float('nan')
        test_f1 = test_result['f1_macro'] if train_result else float('nan')
        val_f1 = val_result['f1_macro'] if val_result else float('nan')
        
        # 计算特征选择比例
        selection_ratio = selected_dim / original_dim if original_dim > 0 else 1.0
        
        # 处理None值，统一转换为字符串'none'用于日志
        feature_selection = 'none' if params.get('feature_selection') is None else params.get('feature_selection')
        selection_mode = 'none' if params.get('selection_mode') is None else params.get('selection_mode')
        regularization = 'none' if params.get('regularization') is None else params.get('regularization')
        normalization = 'none' if params.get('normalization') is None else params.get('normalization')
        
        # 准备日志条目
        log_entry = (f"{experiment_id},{datetime.now().strftime('%Y-%m-%d %H:%M:%S')},"\
                    f"{params.get('apply_pca', False)},{params.get('n_components', 0)},"\
                    f"{normalization},{params.get('class_balance', False)},"\
                    f"{params.get('target_samples', 0)},{regularization},"\
                    f"{params.get('alpha', 0.0)},"\
                    f"{feature_selection},"\
                    f"{selection_mode},"\
                    f"{params.get('selection_threshold', 0.0)},"\
                    f"{params.get('max_features', 0)},"\
                    f"{params.get('l1_ratio', 0.0)},"\
                    f"{train_acc},{test_acc},{val_acc},{train_f1},{test_f1},{val_f1},"\
                    f"{original_dim},{selected_dim},{selection_ratio},"\
                    f"{train_time},{evaluation_time},{status}\n")
        
        # 追加到日志文件
        with open(self.experiment_log_path, 'a') as f:
            f.write(log_entry)
    
    def _convert_numpy_types(self, obj):
    """转换NumPy类型为Python原生类型"""
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    return obj
    
    def generate_summary_report_with_fs(self, top_n=10):
        """
        生成包含特征选择信息的汇总报告
        
        参数:
            top_n: 展示的顶部实验数量
        """
        # 读取实验日志
        if not os.path.exists(self.experiment_log_path):
            self.logger.warning("实验日志不存在", "Experiment log does not exist")
            return
        
        try:
            log_df = pd.read_csv(self.experiment_log_path)
        except:
            self.logger.warning("无法读取实验日志", "Cannot read experiment log")
            return
        
        # 过滤成功的实验
        log_df = log_df[log_df['status'] == 'completed']
        
        if len(log_df) == 0:
            self.logger.warning("没有已完成的实验", "No completed experiments")
            return
        
        # 按测试集准确率排序
        log_df_sorted = log_df.sort_values('test_accuracy', ascending=False)
        
        # 创建报告目录
        report_dir = os.path.join(self.base_dir, "summary_report_with_fs")
        os.makedirs(report_dir, exist_ok=True)
        
        # 保存排序后的实验日志
        log_df_sorted.to_csv(os.path.join(report_dir, "experiments_sorted.csv"), index=False)
        
        # 生成顶部实验表格
        top_df = log_df_sorted.head(top_n)
        
        # 生成HTML报告
        html_report = f"""
        <html>
        <head>
            <title>Pseudo-Inverse Experiments with Feature Selection Summary Report</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 20px; }}
                h1 {{ color: #333366; }}
                h2 {{ color: #666699; }}
                table {{ border-collapse: collapse; width: 100%; }}
                th, td {{ padding: 8px; text-align: left; border-bottom: 1px solid #ddd; }}
                th {{ background-color: #f2f2f2; }}
                tr:hover {{ background-color: #f5f5f5; }}
                .chart-container {{ display: flex; flex-wrap: wrap; justify-content: space-between; }}
                .chart {{ margin: 10px; max-width: 600px; }}
            </style>
        </head>
        <body>
            <h1>Pseudo-Inverse Experiments with Feature Selection Summary Report</h1>
            <p>Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
            <p>Total experiments: {len(log_df)}</p>
            
            <h2>Top {top_n} Experiments by Test Accuracy</h2>
            <table>
                <tr>
                    <th>Experiment ID</th>
                    <th>Test Accuracy</th>
                    <th>Test F1</th>
                    <th>Val Accuracy</th>
                    <th>PCA</th>
                    <th>Feature Selection</th>
                    <th>Original Features</th>
                    <th>Selected Features</th>
                    <th>Selection Ratio</th>
                </tr>
        """
        
        for _, row in top_df.iterrows():
            html_report += f"""
                <tr>
                    <td>{row['experiment_id']}</td>
                    <td>{row['test_accuracy']:.4f}</td>
                    <td>{row['test_f1']:.4f}</td>
                    <td>{row['val_accuracy']:.4f}</td>
                    <td>{'Yes' if row['apply_pca'] else 'No'}</td>
                    <td>{row['feature_selection']}</td>
                    <td>{int(row['original_features'])}</td>
                    <td>{int(row['selected_features'])}</td>
                    <td>{row['selection_ratio']:.2f}</td>
                </tr>
            """
        
        html_report += """
            </table>
            
            <h2>Feature Selection Impact Analysis</h2>
            <div class="chart-container">
        """
        
        # 生成特征选择影响可视化
        # 1. 特征选择方法对准确率的影响
        if 'feature_selection' in log_df.columns:
            plt.figure(figsize=(10, 6))
            sns.boxplot(x='feature_selection', y='test_accuracy', data=log_df)
            plt.title('Impact of Feature Selection Method on Test Accuracy')
            plt.xlabel('Feature Selection Method')
            plt.ylabel('Test Accuracy')
            plt.grid(True, axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            fs_impact_path = os.path.join(report_dir, "feature_selection_impact.png")
            plt.savefig(fs_impact_path, dpi=300)
            plt.close()
            
            html_report += f"""
                <div class="chart">
                    <img src="{os.path.relpath(fs_impact_path, self.base_dir)}" alt="Feature Selection Impact" width="100%">
                </div>
            """
        
        # 2. 特征选择比例与准确率的关系
        plt.figure(figsize=(10, 6))
        plt.scatter(log_df['selection_ratio'], log_df['test_accuracy'], alpha=0.7)
        plt.title('Relationship Between Feature Selection Ratio and Test Accuracy')
        plt.xlabel('Feature Selection Ratio')
        plt.ylabel('Test Accuracy')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        ratio_impact_path = os.path.join(report_dir, "selection_ratio_impact.png")
        plt.savefig(ratio_impact_path, dpi=300)
        plt.close()
        
        html_report += f"""
            <div class="chart">
                <img src="{os.path.relpath(ratio_impact_path, self.base_dir)}" alt="Selection Ratio Impact" width="100%">
            </div>
        """
        
        # 3. 使用不同L1比例的比较
        if 'l1_ratio' in log_df.columns:
            plt.figure(figsize=(10, 6))
            sns.lineplot(x='l1_ratio', y='test_accuracy', data=log_df)
            plt.title('Impact of L1 Ratio on Test Accuracy')
            plt.xlabel('L1 Ratio')
            plt.ylabel('Test Accuracy')
            plt.grid(True, linestyle='--', alpha=0.7)
            plt.tight_layout()
            l1_impact_path = os.path.join(report_dir, "l1_ratio_impact.png")
            plt.savefig(l1_impact_path, dpi=300)
            plt.close()
            
            html_report += f"""
                <div class="chart">
                    <img src="{os.path.relpath(l1_impact_path, self.base_dir)}" alt="L1 Ratio Impact" width="100%">
                </div>
            """
        
        # 4. 特征选择前后对比（使用和不使用特征选择）
        with_fs = log_df[log_df['feature_selection'] != 'none']
        without_fs = log_df[log_df['feature_selection'] == 'none']
        
        if len(with_fs) > 0 and len(without_fs) > 0:
            plt.figure(figsize=(12, 6))
            
            # 准确率对比
            plt.subplot(1, 2, 1)
            data = {
                'With FS': with_fs['test_accuracy'].mean(),
                'Without FS': without_fs['test_accuracy'].mean()
            }
            plt.bar(data.keys(), data.values())
            plt.ylabel('Average Test Accuracy')
            plt.title('Accuracy With vs Without Feature Selection')
            plt.grid(True, axis='y', linestyle='--', alpha=0.7)
            
            # 特征数量对比
            plt.subplot(1, 2, 2)
            data = {
                'Original': log_df['original_features'].mean(),
                'After Selection': with_fs['selected_features'].mean()
            }
            plt.bar(data.keys(), data.values())
            plt.ylabel('Average Feature Count')
            plt.title('Feature Count Before vs After Selection')
            plt.grid(True, axis='y', linestyle='--', alpha=0.7)
            
            plt.tight_layout()
            fs_comparison_path = os.path.join(report_dir, "feature_selection_comparison.png")
            plt.savefig(fs_comparison_path, dpi=300)
            plt.close()
            
            html_report += f"""
                <div class="chart">
                    <img src="{os.path.relpath(fs_comparison_path, self.base_dir)}" alt="Feature Selection Comparison" width="100%">
                </div>
            """
        
        html_report += """
            </div>
            
            <h2>Best Experiment Details</h2>
        """
        
        # 获取最佳实验的详细信息
        if len(top_df) > 0:
            best_exp_id = top_df.iloc[0]['experiment_id']
            best_exp_dir = os.path.join(self.base_dir, best_exp_id)
            
            # 复制最佳实验的图表到报告目录
            important_images = [
                'feature_selection_visualization.png',
                'feature_importance.png',
                'weight_distribution.png', 
                'performance_comparison.png', 
                'confusion_matrix_test.png'
            ]
            
            for img_file in important_images:
                img_path = os.path.join(best_exp_dir, img_file)
                if os.path.exists(img_path):
                    dest_path = os.path.join(report_dir, f"best_{img_file}")
                    import shutil
                    shutil.copy(img_path, dest_path)
                    
                    img_title = img_file.replace('_', ' ').replace('.png', '').title()
                    html_report += f"""
                        <div class="chart">
                            <h3>{img_title}</h3>
                            <img src="{os.path.relpath(dest_path, self.base_dir)}" alt="{img_file}" width="100%">
                        </div>
                    """
        
        html_report += """
            <h2>Conclusion and Recommendations</h2>
            <p>Based on the experimental results, here are the key findings related to feature selection:</p>
            <ul>
        """
        
        # 添加特征选择相关结论
        # 特征选择方法比较
        if 'feature_selection' in log_df.columns:
            fs_impact = log_df.groupby('feature_selection')['test_accuracy'].mean()
            best_fs = fs_impact.idxmax()
            html_report += f"""
                <li>Best feature selection method: {best_fs} (Avg. accuracy: {fs_impact[best_fs]:.4f})</li>
            """
        
        # 特征选择比例分析
        avg_ratio = with_fs['selection_ratio'].mean() if len(with_fs) > 0 else 0
        html_report += f"""
            <li>Average feature selection ratio: {avg_ratio:.2f} ({avg_ratio*100:.1f}% of original features)</li>
        """
        
        # 特征选择前后对比
        if len(with_fs) > 0 and len(without_fs) > 0:
            acc_diff = with_fs['test_accuracy'].mean() - without_fs['test_accuracy'].mean()
            html_report += f"""
                <li>Feature selection {'improves' if acc_diff > 0 else 'reduces'} model performance by {abs(acc_diff):.4f} on average</li>
            """
        
        # 最佳特征选择参数
        if len(top_df) > 0 and top_df.iloc[0]['feature_selection'] != 'none':
            best_row = top_df.iloc[0]
            html_report += f"""
                <li>Best feature selection parameters:
                    <ul>
                        <li>Method: {best_row['feature_selection']}</li>
                        <li>Selection mode: {best_row['selection_mode']}</li>
                        <li>Threshold/Max features: {best_row['selection_threshold'] if best_row['selection_mode'] == 'threshold' else best_row['max_features']}</li>
                        <li>L1 ratio: {best_row['l1_ratio']}</li>
                        <li>Feature reduction: from {int(best_row['original_features'])} to {int(best_row['selected_features'])} features ({best_row['selection_ratio']:.2f} ratio)</li>
                    </ul>
                </li>
            """
        
        html_report += """
            </ul>
        </body>
        </html>
        """
        
        # 保存HTML报告
        with open(os.path.join(report_dir, "summary_report_with_fs.html"), 'w') as f:
            f.write(html_report)
        
        self.logger.info(f"特征选择汇总报告已生成: {os.path.join(report_dir, 'summary_report_with_fs.html')}", 
                       f"Feature selection summary report generated: {os.path.join(report_dir, 'summary_report_with_fs.html')}")

In [ ]:
# 更新参数网格以包含特征选择参数
def create_feature_selection_param_grid():
    """创建包含特征选择参数的网格"""
    
    # 基本参数网格
    base_param_grid = {
        'apply_pca': [True, False],
        'n_components': [50, 100],
        'normalization': ['standard', 'minmax', None],
        'class_balance': [False, True],
        'regularization': [None, 'l2'],
        'alpha': [0.001, 0.01, 0.1]
    }
    
    # 特征选择参数网格
    feature_selection_param_grid = {
        'feature_selection': [None, 'lasso', 'elastic_net'],
        'selection_mode': ['threshold', 'fixed'],
        'selection_threshold': [0.001, 0.01, 0.05],
        'max_features': [50, 100, 200],
        'l1_ratio': [0.5, 0.8, 1.0],
        'scaling_before_selection': [True],
        'selection_metric': ['coefficient']
    }
    
    # 合并参数网格
    full_param_grid = {**base_param_grid, **feature_selection_param_grid}
    
    return full_param_grid

# 添加一个更合理的参数采样函数，避免无效组合
def sample_parameter_combinations(param_grid, max_samples=100, random_state=42):
    """
    从参数网格中采样有效的参数组合
    
    参数:
        param_grid: 参数网格
        max_samples: 最大采样数量
        random_state: 随机种子
        
    返回:
        param_combinations: 参数组合列表
    """
    np.random.seed(random_state)
    
    # 提取所有参数可能的值
    param_values = list(param_grid.values())
    param_keys = list(param_grid.keys())
    
    # 计算所有可能组合的数量
    total_combinations = 1
    for values in param_values:
        total_combinations *= len(values)
    
    print(f"总共可能的参数组合数: {total_combinations}")
    
    # 如果总组合数小于最大采样数，返回所有组合
    if total_combinations <= max_samples:
        # 生成所有组合
        all_combinations = []
        
        def generate_combinations(keys, values, current=0, current_params={}):
            if current == len(keys):
                all_combinations.append(current_params.copy())
                return
            
            for value in values[current]:
                current_params[keys[current]] = value
                generate_combinations(keys, values, current + 1, current_params)
        
        generate_combinations(param_keys, param_values)
        return all_combinations
    
    # 采样参数组合
    param_combinations = []
    
    # 首先添加几个基准配置
    # 1. 不使用特征选择的基准配置
    baseline_config = {
        'apply_pca': False,
        'n_components': 50,
        'normalization': 'standard',
        'class_balance': False,
        'regularization': None,
        'alpha': 0.01,
        'feature_selection': None,
        'selection_mode': 'threshold',
        'selection_threshold': 0.01,
        'max_features': 100,
        'l1_ratio': 1.0,
        'scaling_before_selection': True,
        'selection_metric': 'coefficient'
    }
    param_combinations.append(baseline_config)
    
    # 2. 使用LASSO特征选择的基准配置
    lasso_config = baseline_config.copy()
    lasso_config['feature_selection'] = 'lasso'
    param_combinations.append(lasso_config)
    
    # 3. 使用弹性网络特征选择的基准配置
    elastic_net_config = baseline_config.copy()
    elastic_net_config['feature_selection'] = 'elastic_net'
    elastic_net_config['l1_ratio'] = 0.5
    param_combinations.append(elastic_net_config)
    
    # 随机采样其余配置
    remaining_samples = max_samples - len(param_combinations)
    
    # 避免选择无效组合
    while len(param_combinations) < max_samples:
        config = {}
        
        # 采样每个参数
        for key, values in param_grid.items():
            config[key] = np.random.choice(values)
        
        # 修正无效组合
        # 1. 如果不使用特征选择，调整相关参数
        if config['feature_selection'] is None:
            config['selection_mode'] = 'threshold'  # 不影响实际行为
            config['selection_threshold'] = 0.01
            config['max_features'] = 100
            config['l1_ratio'] = 1.0
        
        # 2. 如果使用的是LASSO，设置l1_ratio为1.0
        if config['feature_selection'] == 'lasso':
            config['l1_ratio'] = 1.0
        
        # 3. 如果不使用PCA，调整相关参数
        if not config['apply_pca']:
            config['n_components'] = 50  # 设为默认值，不影响
        
        # 4. 如果selection_mode是threshold，max_features不相关
        if config['selection_mode'] == 'threshold':
            config['max_features'] = 100  # 设为默认值
        
        # 5. 如果selection_mode是fixed，selection_threshold不相关
        if config['selection_mode'] == 'fixed':
            config['selection_threshold'] = 0.01  # 设为默认值
        
        # 检查是否已经存在相同配置
        if config not in param_combinations:
            param_combinations.append(config)
    
    return param_combinations

In [ ]:
# 更新主实验设置以支持特征选择
# 设置数据路径
TRAIN_DIR = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/train"
TEST_DIR = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/test"
VAL_DIR = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/val"

# 创建实验目录
BASE_DIR = "pseudoinverse_with_fs_experiments"
os.makedirs(BASE_DIR, exist_ok=True)

# 创建日志记录器
logger = BilingualLogger(log_dir=os.path.join(BASE_DIR, "logs"), 
                        log_name=f"experiment_with_fs_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")

logger.info("开始伪逆线性模型实验（包含特征选择）", 
           "Starting pseudo-inverse linear model experiments with feature selection")

# 创建数据加载器
data_loader = BrainVoxelDataLoader(TRAIN_DIR, TEST_DIR, VAL_DIR, logger=logger)

# 加载数据
logger.info("加载数据集", "Loading datasets")
data_loader.load_all_data()

# 创建带特征选择的实验管理器
# 特征选择模式: 'global'(全局选择) 或 'per_experiment'(每次实验单独选择)
FEATURE_SELECTION_MODE = 'global'  # 可根据需要修改

experiment_manager = ExperimentManagerWithFeatureSelection(
    data_loader, 
    base_dir=BASE_DIR, 
    logger=logger,
    feature_selection_mode=FEATURE_SELECTION_MODE
)

logger.info(f"实验管理器初始化完成，特征选择模式: {FEATURE_SELECTION_MODE}", 
           f"Experiment manager initialized with feature selection mode: {FEATURE_SELECTION_MODE}")

# 创建参数网格
param_grid = create_feature_selection_param_grid()

# 打印实验配置
logger.info(f"参数网格创建完成，包含 {len(param_grid)} 个参数", 
           f"Parameter grid created with {len(param_grid)} parameters")

# 设置最大实验数量
MAX_EXPERIMENTS = 100  # 根据需要调整

# 从参数网格中采样合理的组合
param_combinations = sample_parameter_combinations(param_grid, max_samples=MAX_EXPERIMENTS)

logger.info(f"采样了 {len(param_combinations)} 个参数组合", 
           f"Sampled {len(param_combinations)} parameter combinations")

In [ ]:
# 运行基准测试实验以验证特征选择功能
logger.info("运行基准测试实验（包含特征选择）", "Running baseline test experiment with feature selection")

# 基准参数 - 不使用特征选择
baseline_params_no_fs = {
    'apply_pca': False,
    'n_components': 50,
    'normalization': 'standard',
    'class_balance': False,
    'regularization': None,
    'alpha': 0.01,
    'feature_selection': None,
    'selection_mode': 'threshold',
    'selection_threshold': 0.01,
    'max_features': 100,
    'l1_ratio': 1.0,
    'scaling_before_selection': True,
    'selection_metric': 'coefficient'
}

# 基准参数 - 使用LASSO特征选择
baseline_params_lasso = {
    'apply_pca': False,
    'n_components': 50,
    'normalization': 'standard',
    'class_balance': False,
    'regularization': None,
    'alpha': 0.01,
    'feature_selection': 'lasso',
    'selection_mode': 'threshold',
    'selection_threshold': 0.01,
    'max_features': 100,
    'l1_ratio': 1.0,
    'scaling_before_selection': True,
    'selection_metric': 'coefficient'
}

# 运行两个基准实验
try:
    # 运行不使用特征选择的基准实验
    logger.info("运行不使用特征选择的基准实验", "Running baseline experiment without feature selection")
    experiment_id_no_fs, results_no_fs = experiment_manager.run_experiment(baseline_params_no_fs)
    
    logger.info(f"基准实验（无特征选择）{experiment_id_no_fs} 完成", 
               f"Baseline experiment (no feature selection) {experiment_id_no_fs} completed")
    
    # 打印主要评估指标
    logger.info(
        f"基准实验（无特征选择）结果:\n"
        f"  训练集准确率: {results_no_fs['train']['accuracy']:.4f}\n"
        f"  测试集准确率: {results_no_fs['test']['accuracy']:.4f}\n"
        f"  验证集准确率: {results_no_fs['val']['accuracy']:.4f}",
        
        f"Baseline experiment (no feature selection) results:\n"
        f"  Training accuracy: {results_no_fs['train']['accuracy']:.4f}\n"
        f"  Test accuracy: {results_no_fs['test']['accuracy']:.4f}\n"
        f"  Validation accuracy: {results_no_fs['val']['accuracy']:.4f}"
    )
    
    # 运行使用LASSO特征选择的基准实验
    logger.info("运行使用LASSO特征选择的基准实验", "Running baseline experiment with LASSO feature selection")
    experiment_id_lasso, results_lasso = experiment_manager.run_experiment(baseline_params_lasso)
    
    logger.info(f"基准实验（LASSO特征选择）{experiment_id_lasso} 完成", 
               f"Baseline experiment (LASSO feature selection) {experiment_id_lasso} completed")
    
    # 打印主要评估指标
    logger.info(
        f"基准实验（LASSO特征选择）结果:\n"
        f"  训练集准确率: {results_lasso['train']['accuracy']:.4f}\n"
        f"  测试集准确率: {results_lasso['test']['accuracy']:.4f}\n"
        f"  验证集准确率: {results_lasso['val']['accuracy']:.4f}\n"
        f"  原始特征维度: {results_lasso['original_dim']}\n"
        f"  选择后特征维度: {results_lasso['selected_dim']}\n"
        f"  特征选择比例: {results_lasso['selected_dim']/results_lasso['original_dim']:.2f}",
        
        f"Baseline experiment (LASSO feature selection) results:\n"
        f"  Training accuracy: {results_lasso['train']['accuracy']:.4f}\n"
        f"  Test accuracy: {results_lasso['test']['accuracy']:.4f}\n"
        f"  Validation accuracy: {results_lasso['val']['accuracy']:.4f}\n"
        f"  Original feature dimension: {results_lasso['original_dim']}\n"
        f"  Selected feature dimension: {results_lasso['selected_dim']}\n"
        f"  Feature selection ratio: {results_lasso['selected_dim']/results_lasso['original_dim']:.2f}"
    )
    
    # 对比两个基准实验结果
    logger.info(
        f"基准实验对比:\n"
        f"  无特征选择 vs LASSO特征选择:\n"
        f"  测试集准确率: {results_no_fs['test']['accuracy']:.4f} vs {results_lasso['test']['accuracy']:.4f}\n"
        f"  准确率差异: {results_lasso['test']['accuracy'] - results_no_fs['test']['accuracy']:.4f}",
        
        f"Baseline experiments comparison:\n"
        f"  No feature selection vs LASSO feature selection:\n"
        f"  Test accuracy: {results_no_fs['test']['accuracy']:.4f} vs {results_lasso['test']['accuracy']:.4f}\n"
        f"  Accuracy difference: {results_lasso['test']['accuracy'] - results_no_fs['test']['accuracy']:.4f}"
    )
    
except Exception as e:
    logger.error(f"基准测试实验失败: {str(e)}", f"Baseline test experiments failed: {str(e)}")
    print(f"基准测试实验失败: {str(e)}")
    # 显示堆栈跟踪以便调试
    import traceback
    traceback.print_exc()

In [ ]:
# 批量运行实验
logger.info(f"开始批量实验，最大实验数: {len(param_combinations)}", 
           f"Starting batch experiments, max experiments: {len(param_combinations)}")

try:
    # 可以根据需要调整实际运行的实验数量
    experiments_to_run = param_combinations
    
    # 打乱组合顺序，避免类似参数连续运行
    np.random.shuffle(experiments_to_run)
    
    completed_experiments = []
    total_start_time = time.time()
    
    for i, params in enumerate(experiments_to_run):
        # 估计剩余时间
        if i > 0:
            elapsed_time = time.time() - total_start_time
            avg_time_per_exp = elapsed_time / i
            remaining_time = avg_time_per_exp * (len(experiments_to_run) - i)
            
            logger.info(
                f"进度: {i}/{len(experiments_to_run)} ({i/len(experiments_to_run)*100:.1f}%), "
                f"预计剩余时间: {remaining_time/60:.1f} 分钟",
                
                f"Progress: {i}/{len(experiments_to_run)} ({i/len(experiments_to_run)*100:.1f}%), "
                f"Estimated time remaining: {remaining_time/60:.1f} minutes"
            )
        
        try:
            logger.info(f"运行实验 {i+1}/{len(experiments_to_run)}, 参数: {params}",
                      f"Running experiment {i+1}/{len(experiments_to_run)}, parameters: {params}")
            
            experiment_id, _ = experiment_manager.run_experiment(params)
            completed_experiments.append(experiment_id)
            
            logger.info(f"实验 {experiment_id} 完成", f"Experiment {experiment_id} completed")
        except Exception as e:
            logger.error(f"实验失败: {str(e)}", f"Experiment failed: {str(e)}")
            # 继续下一个实验
            continue
    
    total_elapsed_time = time.time() - total_start_time
    logger.info(
        f"批量实验完成，共 {len(completed_experiments)}/{len(experiments_to_run)} 个实验成功，"
        f"总耗时: {total_elapsed_time/60:.1f} 分钟",
        
        f"Batch experiments completed, {len(completed_experiments)}/{len(experiments_to_run)} "
        f"experiments succeeded, total time: {total_elapsed_time/60:.1f} minutes"
    )
    
except Exception as e:
    logger.error(f"批量实验过程中出错: {str(e)}", f"Error during batch experiments: {str(e)}")
    print(f"批量实验过程中出错: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:
# 生成结果汇总报告
logger.info("生成结果汇总报告", "Generating summary report")

try:
    # 生成加入特征选择的汇总报告
    experiment_manager.generate_summary_report_with_fs(top_n=20)
    logger.info("汇总报告生成完成", "Summary report generation completed")
except Exception as e:
    logger.error(f"生成汇总报告失败: {str(e)}", f"Failed to generate summary report: {str(e)}")
    print(f"生成汇总报告失败: {str(e)}")
    import traceback
    traceback.print_exc()

# 打印实验完成消息
logger.info("所有实验和分析已完成", "All experiments and analysis completed")
print("\n所有实验和分析已完成。请查看结果目录获取详细报告。")
print(f"结果目录: {os.path.abspath(BASE_DIR)}")

In [ ]:
# 加载实验日志并分析最佳特征选择结果
log_path = os.path.join(BASE_DIR, "experiment_log_with_fs.csv")
if os.path.exists(log_path):
    try:
        log_df = pd.read_csv(log_path)
        
        # 过滤已完成的实验
        log_df = log_df[log_df['status'] == 'completed']
        
        if len(log_df) > 0:
            # 按测试集准确率排序
            log_df_sorted = log_df.sort_values('test_accuracy', ascending=False)
            
            # 获取最佳实验
            best_exp = log_df_sorted.iloc[0]
            best_exp_id = best_exp['experiment_id']
            
            print(f"\n最佳实验 ID: {best_exp_id}")
            print(f"测试集准确率: {best_exp['test_accuracy']:.4f}")
            print(f"验证集准确率: {best_exp['val_accuracy']:.4f}")
            print(f"特征选择方法: {best_exp['feature_selection']}")
            
            if best_exp['feature_selection'] != 'none' and not pd.isna(best_exp['feature_selection']):
                print(f"特征选择模式: {best_exp['selection_mode']}")
                
                if best_exp['selection_mode'] == 'threshold':
                    print(f"特征选择阈值: {best_exp['selection_threshold']}")
                else:
                    print(f"最大特征数量: {best_exp['max_features']}")
                
                print(f"L1比例: {best_exp['l1_ratio']}")
                print(f"原始特征维度: {int(best_exp['original_features'])}")
                print(f"选择后特征维度: {int(best_exp['selected_features'])}")
                print(f"特征选择比例: {best_exp['selection_ratio']:.2f}")
            
            # 分析不同特征选择方法的效果
            if 'feature_selection' in log_df.columns:
                fs_methods = log_df['feature_selection'].unique()
                print("\n不同特征选择方法的平均性能:")
                
                for fs in fs_methods:
                    if pd.isna(fs):
                        fs_name = "None"
                        fs_df = log_df[log_df['feature_selection'].isna()]
                    else:
                        fs_name = fs
                        fs_df = log_df[log_df['feature_selection'] == fs]
                    
                    if len(fs_df) > 0:
                        avg_acc = fs_df['test_accuracy'].mean()
                        avg_ratio = fs_df['selection_ratio'].mean() if 'selection_ratio' in fs_df.columns else 1.0
                        
                        print(f"  {fs_name}: 平均准确率 {avg_acc:.4f}, 平均特征选择比例 {avg_ratio:.2f}, 样本数 {len(fs_df)}")
            
            # 分析特征选择比例与性能的关系
            if 'selection_ratio' in log_df.columns:
                # 将选择比例分为几个区间
                log_df['ratio_bin'] = pd.cut(log_df['selection_ratio'], 
                                            bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0], 
                                            labels=['0-20%', '20-40%', '40-60%', '60-80%', '80-100%'])
                
                ratio_performance = log_df.groupby('ratio_bin')['test_accuracy'].agg(['mean', 'std', 'count'])
                
                print("\n特征选择比例与性能的关系:")
                print(ratio_performance)
                
                # 绘制特征选择比例与性能的关系图
                plt.figure(figsize=(10, 6))
                sns.boxplot(x='ratio_bin', y='test_accuracy', data=log_df)
                plt.title('Relationship Between Feature Selection Ratio and Test Accuracy')
                plt.xlabel('Feature Selection Ratio')
                plt.ylabel('Test Accuracy')
                plt.grid(True, axis='y', linestyle='--', alpha=0.7)
                plt.show()
            
            # 分析最佳特征选择结果
            best_exp_dir = os.path.join(BASE_DIR, best_exp_id)
            
            # 加载最佳特征选择器（如果存在）
            feature_selector_path = os.path.join(best_exp_dir, "feature_selector.pkl")
            
            if os.path.exists(feature_selector_path) and best_exp['feature_selection'] != 'none' and not pd.isna(best_exp['feature_selection']):
                try:
                    selector = FeatureSelector()
                    selector.load(feature_selector_path)
                    
                    # 显示特征重要性
                    feature_importance = selector.get_feature_importance()
                    selected_indices = selector.get_selected_indices()
                    
                    print(f"\n特征重要性分析:")
                    print(f"  特征总数: {len(feature_importance)}")
                    print(f"  选择的特征数: {len(selected_indices)}")
                    
                    # 显示最重要的10个特征
                    top_n = 10
                    top_indices = np.argsort(feature_importance)[-top_n:][::-1]
                    
                    print(f"\n前{top_n}个最重要的特征:")
                    for i, idx in enumerate(top_indices):
                        print(f"  特征 {idx}: 重要性 {feature_importance[idx]:.4f}")
                    
                    # 可视化选择的特征
                    plt.figure(figsize=(12, 6))
                    
                    # 特征重要性分布
                    plt.subplot(1, 2, 1)
                    plt.hist(feature_importance, bins=50, alpha=0.7)
                    plt.axvline(x=selector.selection_threshold if selector.selection_mode == 'threshold' else 
                               feature_importance[np.argsort(feature_importance)[-selector.max_features]], 
                               color='r', linestyle='--', 
                               label=f"Selection Threshold")
                    plt.xlabel('Feature Importance')
                    plt.ylabel('Frequency')
                    plt.title('Distribution of Feature Importance')
                    plt.legend()
                    plt.grid(True, linestyle='--', alpha=0.7)
                    
                    # 选择的特征
                    plt.subplot(1, 2, 2)
                    selected_mask = np.zeros(len(feature_importance), dtype=bool)
                    selected_mask[selected_indices] = True
                    
                    plt.scatter(range(len(feature_importance)), feature_importance, alpha=0.5, 
                               c=['blue' if s else 'gray' for s in selected_mask])
                    plt.xlabel('Feature Index')
                    plt.ylabel('Feature Importance')
                    plt.title('Selected Features in Feature Space')
                    plt.grid(True, linestyle='--', alpha=0.7)
                    
                    plt.tight_layout()
                    plt.show()
                    
                except Exception as e:
                    print(f"加载特征选择器失败: {str(e)}")
    
    except Exception as e:
        print(f"加载和分析实验结果失败: {str(e)}")
        import traceback
        traceback.print_exc()
else:
    print(f"实验日志文件不存在: {log_path}")